In [1]:
# ============================================================
# NOTEBOOK 6: RAG PIPELINE
# ============================================================
# Project:
# Healthcare Retrieval-Augmented Generation (RAG) System
#
# Purpose:
# Build the complete RAG pipeline:
#
#   User Question
#        ↓
#   Query Embedding
#        ↓
#   FAISS Retrieval
#        ↓
#   NHS Context
#        ↓
#   Prompt Construction
#        ↓
#   Llama 3.1 8B Instruct
#        ↓
#   Final Answer
#
# The four embedding models will remain available so that
# retrieval can later be compared experimentally.
# ============================================================


# ============================================================
# STEP 1: MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ============================================================
# STEP 2: INSTALL REQUIRED LIBRARIES
# ============================================================
# sentence-transformers:
#   Used to convert user questions into query embeddings using
#   MiniLM, MPNet, E5, and BGE.
#
# faiss-cpu:
#   Used to search the FAISS indexes created in Notebook 5.
#
# transformers:
#   Used to load and run Llama 3.1 8B Instruct.
#
# accelerate:
#   Helps Transformers efficiently place the LLM on the
#   available Colab hardware.
#
# bitsandbytes:
#   Allows us to use quantized model loading to reduce GPU
#   memory requirements when running Llama 3.1 8B.
# ============================================================

!pip install -q \
    sentence-transformers \
    faiss-cpu \
    transformers \
    accelerate \
    bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.2 MB/s eta 0:00:00


In [3]:
# ============================================================
# STEP 3: IMPORT REQUIRED LIBRARIES
# ============================================================
# Path:
#   Manages project directories and file paths.
#
# JSON:
#   Loads FAISS metadata containing the NHS text chunks.
#
# Time:
#   Measures retrieval and generation performance.
#
# NumPy:
#   Handles embedding vectors.
#
# FAISS:
#   Performs vector similarity search.
#
# SentenceTransformer:
#   Converts user questions into embeddings.
#
# Transformers:
#   Will later load Llama 3.1 8B Instruct.
# ============================================================

from pathlib import Path
import json
import time

import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

print("Core RAG libraries imported successfully.")

Core RAG libraries imported successfully.


In [4]:
# ============================================================
# STEP 4: DEFINE RAG PROJECT PATHS
# ============================================================
# This notebook will use:
#
#   data/faiss/
#       MiniLM/
#           faiss_index.bin
#           metadata.json
#
#       MPNet/
#           faiss_index.bin
#           metadata.json
#
#       E5/
#           faiss_index.bin
#           metadata.json
#
#       BGE/
#           faiss_index.bin
#           metadata.json
#
# Later, the selected FAISS index will retrieve relevant NHS
# chunks for a user's question.
# ============================================================


# ------------------------------------------------------------
# MAIN PROJECT DIRECTORY
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Healthcare_RAG"
)


# ------------------------------------------------------------
# FAISS DIRECTORY
# ------------------------------------------------------------

FAISS_DIR = (
    PROJECT_DIR
    / "data"
    / "faiss"
)


# ------------------------------------------------------------
# RESULTS DIRECTORY
# ------------------------------------------------------------
# Retrieval experiments and report outputs will eventually
# be stored here.

RESULTS_DIR = (
    PROJECT_DIR
    / "results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# EMBEDDING MODELS
# ------------------------------------------------------------
# These must correspond to the models used in Notebook 4.

EMBEDDING_MODELS = {

    "MiniLM":
        "sentence-transformers/all-MiniLM-L6-v2",

    "MPNet":
        "sentence-transformers/all-mpnet-base-v2",

    "E5":
        "intfloat/multilingual-e5-base",

    "BGE":
        "BAAI/bge-small-en-v1.5"

}


# ------------------------------------------------------------
# MODEL ORDER
# ------------------------------------------------------------

MODEL_NAMES = [
    "MiniLM",
    "MPNet",
    "E5",
    "BGE"
]


# ------------------------------------------------------------
# DISPLAY PATHS
# ------------------------------------------------------------

print("=" * 70)
print("RAG PROJECT PATHS")
print("=" * 70)

print(f"Project Directory : {PROJECT_DIR}")
print(f"FAISS Directory   : {FAISS_DIR}")
print(f"Results Directory : {RESULTS_DIR}")

print("\nEmbedding Models:")

for model_name, model_id in EMBEDDING_MODELS.items():

    print(f"{model_name:10} : {model_id}")

print("=" * 70)

RAG PROJECT PATHS
Project Directory : /content/drive/MyDrive/Healthcare_RAG
FAISS Directory   : /content/drive/MyDrive/Healthcare_RAG/data/faiss
Results Directory : /content/drive/MyDrive/Healthcare_RAG/results

Embedding Models:
MiniLM     : sentence-transformers/all-MiniLM-L6-v2
MPNet      : sentence-transformers/all-mpnet-base-v2
E5         : intfloat/multilingual-e5-base
BGE        : BAAI/bge-small-en-v1.5


In [5]:
# ============================================================
# STEP 5: LOAD FAISS INDEXES AND METADATA
# ============================================================
# This cell loads:
#
#   FAISS vector index
#   +
#   Corresponding NHS metadata
#
# for all four embedding models.
#
# The metadata is essential because FAISS returns vector IDs,
# while the metadata gives us the actual NHS text and source.
# ============================================================


# ------------------------------------------------------------
# Dictionaries for loaded indexes and metadata
# ------------------------------------------------------------

faiss_indexes = {}

faiss_metadata = {}


print("=" * 70)
print("LOADING FAISS INDEXES AND METADATA")
print("=" * 70)


# ------------------------------------------------------------
# Load each model's index and metadata
# ------------------------------------------------------------

for model_name in MODEL_NAMES:

    print("\n" + "-" * 70)
    print(f"LOADING: {model_name}")
    print("-" * 70)

    # --------------------------------------------------------
    # File paths
    # --------------------------------------------------------

    model_dir = FAISS_DIR / model_name

    index_path = model_dir / "faiss_index.bin"

    metadata_path = model_dir / "metadata.json"


    # --------------------------------------------------------
    # Check files exist
    # --------------------------------------------------------

    if not index_path.exists():

        raise FileNotFoundError(
            f"FAISS index not found: {index_path}"
        )

    if not metadata_path.exists():

        raise FileNotFoundError(
            f"Metadata file not found: {metadata_path}"
        )


    # --------------------------------------------------------
    # Load FAISS index
    # --------------------------------------------------------

    index = faiss.read_index(
        str(index_path)
    )


    # --------------------------------------------------------
    # Load metadata
    # --------------------------------------------------------

    with open(
        metadata_path,
        "r",
        encoding="utf-8"
    ) as file:

        metadata = json.load(file)


    # --------------------------------------------------------
    # Store in dictionaries
    # --------------------------------------------------------

    faiss_indexes[model_name] = index

    faiss_metadata[model_name] = metadata


    # --------------------------------------------------------
    # Display information
    # --------------------------------------------------------

    print(f"Vectors   : {index.ntotal}")
    print(f"Dimension : {index.d}")
    print(f"Metadata  : {len(metadata)} records")


print("\n")
print("=" * 70)
print("ALL FAISS INDEXES AND METADATA LOADED")
print("=" * 70)

LOADING FAISS INDEXES AND METADATA

----------------------------------------------------------------------
LOADING: MiniLM
----------------------------------------------------------------------
Vectors   : 3796
Dimension : 384
Metadata  : 3796 records

----------------------------------------------------------------------
LOADING: MPNet
----------------------------------------------------------------------
Vectors   : 3796
Dimension : 768
Metadata  : 3796 records

----------------------------------------------------------------------
LOADING: E5
----------------------------------------------------------------------
Vectors   : 3796
Dimension : 768
Metadata  : 3796 records

----------------------------------------------------------------------
LOADING: BGE
----------------------------------------------------------------------
Vectors   : 3796
Dimension : 384
Metadata  : 3796 records


ALL FAISS INDEXES AND METADATA LOADED


In [6]:
# ============================================================
# STEP 6: CREATE THE SEMANTIC RETRIEVAL FUNCTION
# ============================================================
# This function performs the RETRIEVAL part of RAG.
#
# Input:
#   - User question
#   - Embedding model name
#   - Number of results (Top-K)
#
# Process:
#   1. Load the appropriate embedding model.
#   2. Convert the question into a query embedding.
#   3. Search the corresponding FAISS index.
#   4. Retrieve the matching NHS metadata.
#
# Output:
#   A list containing the retrieved NHS chunks, their sources,
#   ranks, and similarity scores.
#
# E5 SPECIAL HANDLING:
#   E5 expects:
#       query: <user question>
#
#   The other three models use the question directly.
# ============================================================


def retrieve_documents(
    question,
    model_name="BGE",
    top_k=5
):

    # --------------------------------------------------------
    # Validate model name
    # --------------------------------------------------------

    if model_name not in MODEL_NAMES:

        raise ValueError(
            f"Unknown model: {model_name}. "
            f"Choose from: {MODEL_NAMES}"
        )


    # --------------------------------------------------------
    # Load the corresponding embedding model
    # --------------------------------------------------------

    model_id = EMBEDDING_MODELS[model_name]

    query_model = SentenceTransformer(
        model_id
    )


    # --------------------------------------------------------
    # Prepare the query
    #
    # E5 requires the "query:" prefix.
    # --------------------------------------------------------

    if model_name == "E5":

        query_text = f"query: {question}"

    else:

        query_text = question


    # --------------------------------------------------------
    # Generate query embedding
    # --------------------------------------------------------

    query_embedding = query_model.encode(
        [query_text],
        normalize_embeddings=True
    )


    # --------------------------------------------------------
    # Convert to FAISS-compatible NumPy format
    # --------------------------------------------------------

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )


    # --------------------------------------------------------
    # Retrieve the appropriate FAISS index
    # --------------------------------------------------------

    index = faiss_indexes[model_name]

    metadata = faiss_metadata[model_name]


    # --------------------------------------------------------
    # Perform similarity search
    # --------------------------------------------------------

    scores, indices = index.search(
        query_embedding,
        top_k
    )


    # --------------------------------------------------------
    # Construct retrieval results
    # --------------------------------------------------------

    results = []

    for rank, (score, index_id) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        result = metadata[index_id]

        results.append({

            "rank": rank,

            "score": float(score),

            "chunk_id": result["chunk_id"],

            "source_file": result["source_file"],

            "text": result["text"]

        })


    # --------------------------------------------------------
    # Return retrieved documents
    # --------------------------------------------------------

    return results

In [7]:
# ============================================================
# STEP 14A: VERIFY GPU BEFORE LOADING LLAMA
# ============================================================

import torch

print("=" * 70)
print("GPU VERIFICATION")
print("=" * 70)

if not torch.cuda.is_available():

    raise RuntimeError(
        "GPU is not available. "
        "Go to Runtime > Change runtime type > T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)

gpu_memory = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)

print(f"CUDA Available : True")
print(f"GPU            : {gpu_name}")
print(f"GPU Memory     : {gpu_memory:.2f} GB")

print("=" * 70)
print("GPU READY")
print("=" * 70)

GPU VERIFICATION
CUDA Available : True
GPU            : Tesla T4
GPU Memory     : 14.56 GB
GPU READY


In [8]:
# ============================================================
# STEP 7: TEST SEMANTIC RETRIEVAL
# ============================================================
# This cell tests the retrieval function created in Cell 6.
#
# We will:
#   1. Ask a healthcare question.
#   2. Search the BGE FAISS index.
#   3. Retrieve the top 5 NHS chunks.
#   4. Display their similarity scores and source articles.
#
# This is a pipeline test, NOT the formal retrieval evaluation.
# ============================================================

test_question = "What are the symptoms of asthma?"

TOP_K = 5

print("=" * 70)
print("SEMANTIC RETRIEVAL TEST")
print("=" * 70)

print(f"Question : {test_question}")
print(f"Model    : BGE")
print(f"Top-K    : {TOP_K}")

# ------------------------------------------------------------
# Retrieve relevant NHS chunks
# ------------------------------------------------------------

retrieved_results = retrieve_documents(
    question=test_question,
    model_name="BGE",
    top_k=TOP_K
)

# ------------------------------------------------------------
# Display retrieved results
# ------------------------------------------------------------

for result in retrieved_results:

    print("\n" + "-" * 70)

    print(f"Rank       : {result['rank']}")
    print(f"Similarity : {result['score']:.4f}")
    print(f"Source     : {result['source_file']}")
    print(f"Chunk ID   : {result['chunk_id']}")

    print("\nRetrieved Text:")
    print(result["text"][:700])


print("\n")
print("=" * 70)
print("RETRIEVAL TEST COMPLETED")
print("=" * 70)

SEMANTIC RETRIEVAL TEST
Question : What are the symptoms of asthma?
Model    : BGE
Top-K    : 5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


----------------------------------------------------------------------
Rank       : 1
Similarity : 0.8265
Source     : asthma.txt
Chunk ID   : 1

Retrieved Text:
# Asthma

Asthma is a common condition that affects your breathing. It cannot currently be cured, but if it's well treated you should not have problems with symptoms.

## Symptoms of asthma

The main symptoms of asthma are problems with breathing, such as:

• wheezing

• coughing

• shortness of breath

• your chest feeling tight

Asthma symptoms can be mild or severe. They usually come and go. They may be worse at night and early in the morning.

An asthma attack is when you have severe symptoms and it's difficult to breathe, which can be life-threatening.

Symptoms can be triggered by different things including exercise, high levels of air pollution, cold air, or contact with something you

----------------------------------------------------------------------
Rank       : 2
Similarity : 0.8027
Source     : asthma.txt
Chunk

In [9]:
# ============================================================
# STEP 8: LOAD QUERY-EMBEDDING MODELS ONCE
# ============================================================
# Loading a model repeatedly for every user question is
# inefficient.
#
# Instead, we load each embedding model once and keep the
# models available in memory.
#
# Later:
#
# User Question
#      ↓
# Existing Embedding Model
#      ↓
# Query Vector
#      ↓
# FAISS
#
# This is much more suitable for an interactive chatbot.
# ============================================================

# ------------------------------------------------------------
# Dictionary to store loaded query models
# ------------------------------------------------------------

query_models = {}

print("=" * 70)
print("LOADING QUERY-EMBEDDING MODELS")
print("=" * 70)

# ------------------------------------------------------------
# Load each model once
# ------------------------------------------------------------

for model_name, model_id in EMBEDDING_MODELS.items():

    print("\n" + "-" * 70)
    print(f"Loading: {model_name}")
    print(f"Model ID: {model_id}")
    print("-" * 70)

    query_models[model_name] = SentenceTransformer(
        model_id
    )

    print(f"{model_name} loaded successfully.")


# ------------------------------------------------------------
# Display final status
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("ALL QUERY-EMBEDDING MODELS LOADED")
print("=" * 70)

for model_name in query_models:

    dimension = (
        query_models[model_name]
        .get_sentence_embedding_dimension()
    )

    print(
        f"{model_name:10} : "
        f"{dimension} dimensions"
    )

print("=" * 70)

LOADING QUERY-EMBEDDING MODELS

----------------------------------------------------------------------
Loading: MiniLM
Model ID: sentence-transformers/all-MiniLM-L6-v2
----------------------------------------------------------------------


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (37 outputs left out)

E5 loaded successfully.

----------------------------------------------------------------------
Loading: BGE
Model ID: BAAI/bge-small-en-v1.5
----------------------------------------------------------------------


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BGE loaded successfully.


ALL QUERY-EMBEDDING MODELS LOADED
MiniLM     : 384 dimensions
MPNet      : 768 dimensions
E5         : 768 dimensions
BGE        : 384 dimensions


/tmp/ipykernel_693/1250551402.py:64: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  .get_sentence_embedding_dimension()


In [10]:
# ============================================================
# STEP 9: CREATE OPTIMIZED RETRIEVAL FUNCTION
# ============================================================
# This function performs the retrieval stage of the RAG system.
#
# It does NOT reload the embedding model.
# The models loaded in Cell 8 are reused.
#
# Input:
#   - User question
#   - Embedding model
#   - Top-K value
#
# Output:
#   - Retrieved NHS chunks
#   - Similarity scores
#   - Source articles
#   - Chunk IDs
#
# E5:
#   Uses "query:" because E5 distinguishes queries from
#   document passages.
# ============================================================


def retrieve_documents(
    question,
    model_name="BGE",
    top_k=5
):

    # --------------------------------------------------------
    # Validate model name
    # --------------------------------------------------------

    if model_name not in MODEL_NAMES:

        raise ValueError(
            f"Unknown model: {model_name}. "
            f"Available models: {MODEL_NAMES}"
        )


    # --------------------------------------------------------
    # Get the already-loaded embedding model
    # --------------------------------------------------------

    query_model = query_models[model_name]


    # --------------------------------------------------------
    # Prepare the query
    # --------------------------------------------------------

    if model_name == "E5":

        query_text = f"query: {question}"

    else:

        query_text = question


    # --------------------------------------------------------
    # Generate query embedding
    # --------------------------------------------------------

    query_embedding = query_model.encode(
        [query_text],
        normalize_embeddings=True
    )


    # --------------------------------------------------------
    # Convert to FAISS-compatible format
    # --------------------------------------------------------

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )


    # --------------------------------------------------------
    # Retrieve the corresponding FAISS index
    # --------------------------------------------------------

    index = faiss_indexes[model_name]

    metadata = faiss_metadata[model_name]


    # --------------------------------------------------------
    # Perform similarity search
    # --------------------------------------------------------

    scores, indices = index.search(
        query_embedding,
        top_k
    )


    # --------------------------------------------------------
    # Convert FAISS results into readable records
    # --------------------------------------------------------

    results = []

    for rank, (score, index_id) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):

        result = metadata[index_id]

        results.append({

            "rank": rank,

            "score": float(score),

            "chunk_id": result["chunk_id"],

            "source_file": result["source_file"],

            "text": result["text"]

        })


    # --------------------------------------------------------
    # Return retrieved NHS documents
    # --------------------------------------------------------

    return results

In [11]:
# ============================================================
# STEP 10: TEST OPTIMIZED RETRIEVAL
# ============================================================
# This cell tests the optimized retrieval function created in
# Cell 9.
#
# We will:
#   1. Ask a healthcare question.
#   2. Use the already-loaded BGE model.
#   3. Search the BGE FAISS index.
#   4. Retrieve the top 5 NHS chunks.
#   5. Display their scores, sources, and text.
#
# This confirms that the retrieval component is ready to be
# connected to the LLM.
# ============================================================

test_question = "What are the symptoms of asthma?"

TOP_K = 5

print("=" * 70)
print("OPTIMIZED RETRIEVAL TEST")
print("=" * 70)

print(f"Question : {test_question}")
print(f"Model    : BGE")
print(f"Top-K    : {TOP_K}")

# ------------------------------------------------------------
# Perform retrieval
# ------------------------------------------------------------

retrieved_results = retrieve_documents(
    question=test_question,
    model_name="BGE",
    top_k=TOP_K
)

# ------------------------------------------------------------
# Display retrieved results
# ------------------------------------------------------------

for result in retrieved_results:

    print("\n" + "-" * 70)

    print(f"Rank       : {result['rank']}")
    print(f"Similarity : {result['score']:.4f}")
    print(f"Source     : {result['source_file']}")
    print(f"Chunk ID   : {result['chunk_id']}")

    print("\nRetrieved Text:")
    print(result["text"][:700])


print("\n")
print("=" * 70)
print("OPTIMIZED RETRIEVAL TEST COMPLETED")
print("=" * 70)

OPTIMIZED RETRIEVAL TEST
Question : What are the symptoms of asthma?
Model    : BGE
Top-K    : 5

----------------------------------------------------------------------
Rank       : 1
Similarity : 0.8265
Source     : asthma.txt
Chunk ID   : 1

Retrieved Text:
# Asthma

Asthma is a common condition that affects your breathing. It cannot currently be cured, but if it's well treated you should not have problems with symptoms.

## Symptoms of asthma

The main symptoms of asthma are problems with breathing, such as:

• wheezing

• coughing

• shortness of breath

• your chest feeling tight

Asthma symptoms can be mild or severe. They usually come and go. They may be worse at night and early in the morning.

An asthma attack is when you have severe symptoms and it's difficult to breathe, which can be life-threatening.

Symptoms can be triggered by different things including exercise, high levels of air pollution, cold air, or contact with something you

--------------------------------------

In [12]:
# ============================================================
# STEP 11: MEASURE RETRIEVAL TIME FOR ALL FOUR MODELS
# ============================================================
# This cell measures how long each model takes to:
#
#   1. Convert the question into an embedding.
#   2. Search its FAISS index.
#
# We use the SAME question and SAME Top-K value for every
# model to make the timing comparison consistent.
#
# IMPORTANT:
# This is a performance measurement, not a retrieval-quality
# metric.
# ============================================================

test_question = "What are the symptoms of asthma?"

TOP_K = 5

retrieval_timing_results = []

print("=" * 70)
print("RETRIEVAL LATENCY EXPERIMENT")
print("=" * 70)

print(f"Question : {test_question}")
print(f"Top-K    : {TOP_K}")


# ------------------------------------------------------------
# Test every embedding model
# ------------------------------------------------------------

for model_name in MODEL_NAMES:

    print("\n" + "-" * 70)
    print(f"Testing: {model_name}")
    print("-" * 70)

    # --------------------------------------------------------
    # Start timer
    # --------------------------------------------------------

    start_time = time.time()

    # --------------------------------------------------------
    # Perform retrieval
    # --------------------------------------------------------

    results = retrieve_documents(
        question=test_question,
        model_name=model_name,
        top_k=TOP_K
    )

    # --------------------------------------------------------
    # Stop timer
    # --------------------------------------------------------

    elapsed_time = time.time() - start_time

    # --------------------------------------------------------
    # Record result
    # --------------------------------------------------------

    retrieval_timing_results.append({

        "Model": model_name,

        "Top_K": TOP_K,

        "Retrieval_Time_Seconds": round(
            elapsed_time,
            4
        ),

        "Top_Result_Source": results[0]["source_file"],

        "Top_Result_Score": round(
            results[0]["score"],
            4
        )

    })

    print(
        f"Retrieval Time : "
        f"{elapsed_time:.4f} seconds"
    )

    print(
        f"Top Source     : "
        f"{results[0]['source_file']}"
    )

    print(
        f"Top Score      : "
        f"{results[0]['score']:.4f}"
    )


# ------------------------------------------------------------
# Display final results
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("RETRIEVAL LATENCY RESULTS")
print("=" * 70)

import pandas as pd

retrieval_timing_df = pd.DataFrame(
    retrieval_timing_results
)

display(retrieval_timing_df)

print("=" * 70)

RETRIEVAL LATENCY EXPERIMENT
Question : What are the symptoms of asthma?
Top-K    : 5

----------------------------------------------------------------------
Testing: MiniLM
----------------------------------------------------------------------
Retrieval Time : 0.0653 seconds
Top Source     : asthma.txt
Top Score      : 0.7398

----------------------------------------------------------------------
Testing: MPNet
----------------------------------------------------------------------
Retrieval Time : 0.2668 seconds
Top Source     : asthma.txt
Top Score      : 0.7600

----------------------------------------------------------------------
Testing: E5
----------------------------------------------------------------------
Retrieval Time : 0.0445 seconds
Top Source     : asthma.txt
Top Score      : 0.8932

----------------------------------------------------------------------
Testing: BGE
----------------------------------------------------------------------
Retrieval Time : 0.0107 seconds
To

,Model,Top_K,Retrieval_Time_Seconds,Top_Result_Source,Top_Result_Score
0,MiniLM,5,0.0653,asthma.txt,0.7398
1,MPNet,5,0.2668,asthma.txt,0.7600
2,E5,5,0.0445,asthma.txt,0.8932
3,BGE,5,0.0107,asthma.txt,0.8265


In [13]:
# ============================================================
# STEP 12: BUILD RAG CONTEXT FROM RETRIEVED DOCUMENTS
# ============================================================
# This cell converts the FAISS retrieval results into a single
# structured context block.
#
# Input:
#   Retrieved NHS chunks from FAISS
#
# Output:
#   A formatted context string containing:
#       - Rank
#       - NHS source
#       - Chunk ID
#       - Retrieved text
#
# This context will later be inserted into the LLM prompt.
#
# IMPORTANT:
#   We are NOT generating an answer yet.
#   We are only preparing the evidence that the LLM will use.
# ============================================================


def build_context(retrieved_results):
    """
    Combine retrieved NHS chunks into a structured context
    string for the RAG prompt.
    """

    context_parts = []

    # --------------------------------------------------------
    # Process every retrieved result
    # --------------------------------------------------------

    for result in retrieved_results:

        context_block = (
            f"[Source {result['rank']}]\n"
            f"NHS Source: {result['source_file']}\n"
            f"Chunk ID: {result['chunk_id']}\n"
            f"Content:\n"
            f"{result['text']}\n"
        )

        context_parts.append(
            context_block
        )

    # --------------------------------------------------------
    # Combine all retrieved chunks
    # --------------------------------------------------------

    context = "\n" + ("-" * 70 + "\n").join(
        context_parts
    )

    return context


# ============================================================
# TEST CONTEXT CONSTRUCTION
# ============================================================

test_question = "What are the symptoms of asthma?"

retrieved_results = retrieve_documents(
    question=test_question,
    model_name="BGE",
    top_k=5
)

context = build_context(
    retrieved_results
)


# ------------------------------------------------------------
# Display the generated context
# ------------------------------------------------------------

print("=" * 70)
print("RAG CONTEXT")
print("=" * 70)

print(context)

print("\n")
print("=" * 70)
print("CONTEXT CONSTRUCTION COMPLETED")
print("=" * 70)

RAG CONTEXT

[Source 1]
NHS Source: asthma.txt
Chunk ID: 1
Content:
# Asthma

Asthma is a common condition that affects your breathing. It cannot currently be cured, but if it's well treated you should not have problems with symptoms.

## Symptoms of asthma

The main symptoms of asthma are problems with breathing, such as:

• wheezing

• coughing

• shortness of breath

• your chest feeling tight

Asthma symptoms can be mild or severe. They usually come and go. They may be worse at night and early in the morning.

An asthma attack is when you have severe symptoms and it's difficult to breathe, which can be life-threatening.

Symptoms can be triggered by different things including exercise, high levels of air pollution, cold air, or contact with something you're allergic to, such as pollen, dust, mould or animals.

Asthma usually starts in children, but it can happen at any age.
----------------------------------------------------------------------
[Source 2]
NHS Source: asthma.txt
Chun

In [14]:
# ============================================================
# STEP 13: CREATE THE RAG PROMPT TEMPLATE
# ============================================================
# This prompt will later be given to Llama 3.1 8B Instruct.
#
# The LLM receives:
#   1. The user's question.
#   2. Retrieved NHS context.
#
# The instructions require the model to:
#   - Use the retrieved NHS context as the primary evidence.
#   - Avoid inventing unsupported information.
#   - Clearly state when the retrieved context does not contain
#     enough information.
#   - Provide the NHS source used for the answer.
#
# IMPORTANT:
#   This is a prompt template only.
#   Llama has NOT been loaded or executed yet.
# ============================================================


def create_rag_prompt(
    question,
    context
):
    """
    Construct the prompt that will be supplied to the LLM.
    """

    prompt = f"""
You are a healthcare information assistant using
retrieved information from NHS sources.

Your task is to answer the user's question using ONLY
the information provided in the retrieved NHS context.

Rules:

1. Base your answer on the retrieved NHS information.
2. Do not invent facts that are not supported by the context.
3. Do not make a diagnosis.
4. Do not present yourself as a doctor or healthcare professional.
5. If the retrieved context does not contain enough information
   to answer the question, clearly say that the available NHS
   information does not provide enough information.
6. Keep the answer clear and easy to understand.
7. When relevant, mention the NHS source article used.
8. If the retrieved NHS context contains emergency guidance,
   preserve that guidance accurately.
9. Do not replace NHS emergency instructions with your own
   medical instructions.

USER QUESTION:
{question}

RETRIEVED NHS CONTEXT:
{context}

ANSWER:
"""

    return prompt


# ============================================================
# TEST THE PROMPT
# ============================================================

test_prompt = create_rag_prompt(
    question=test_question,
    context=context
)


print("=" * 70)
print("RAG PROMPT CREATED")
print("=" * 70)

print(test_prompt[:5000])

print("\n")
print("=" * 70)
print("PROMPT TEMPLATE TEST COMPLETED")
print("=" * 70)

RAG PROMPT CREATED

You are a healthcare information assistant using
retrieved information from NHS sources.

Your task is to answer the user's question using ONLY
the information provided in the retrieved NHS context.

Rules:

1. Base your answer on the retrieved NHS information.
2. Do not invent facts that are not supported by the context.
3. Do not make a diagnosis.
4. Do not present yourself as a doctor or healthcare professional.
5. If the retrieved context does not contain enough information
   to answer the question, clearly say that the available NHS
   information does not provide enough information.
6. Keep the answer clear and easy to understand.
7. When relevant, mention the NHS source article used.
8. If the retrieved NHS context contains emergency guidance,
   preserve that guidance accurately.
9. Do not replace NHS emergency instructions with your own
   medical instructions.

USER QUESTION:
What are the symptoms of asthma?

RETRIEVED NHS CONTEXT:

[Source 1]
NHS Source:

In [15]:
# ============================================================
# STEP 15: CHECK COLAB GPU RESOURCES
# ============================================================
# This cell checks:
#   - Whether a GPU is available
#   - GPU name
#   - Total GPU memory
#   - Currently allocated GPU memory
#   - Currently reserved GPU memory
#
# We will use this information to choose the generator model
# that can actually run within today's Colab environment.
#
# IMPORTANT:
# Do NOT download another large model before checking this.
# ============================================================

import torch

print("=" * 70)
print("COLAB GPU RESOURCE CHECK")
print("=" * 70)

# ------------------------------------------------------------
# Check whether CUDA is available
# ------------------------------------------------------------

if torch.cuda.is_available():

    print("CUDA Available : True")

    # --------------------------------------------------------
    # Get GPU information
    # --------------------------------------------------------

    gpu_name = torch.cuda.get_device_name(0)

    total_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    allocated_memory = (
        torch.cuda.memory_allocated(0)
        / (1024 ** 3)
    )

    reserved_memory = (
        torch.cuda.memory_reserved(0)
        / (1024 ** 3)
    )

    # --------------------------------------------------------
    # Display GPU information
    # --------------------------------------------------------

    print(f"GPU Name      : {gpu_name}")
    print(f"Total VRAM    : {total_memory:.2f} GB")
    print(f"Allocated VRAM: {allocated_memory:.2f} GB")
    print(f"Reserved VRAM : {reserved_memory:.2f} GB")

    # --------------------------------------------------------
    # Basic assessment
    # --------------------------------------------------------

    if total_memory >= 14:

        print("\nGPU Memory Assessment:")
        print("Sufficient for attempting an 8B 4-bit model.")

    elif total_memory >= 8:

        print("\nGPU Memory Assessment:")
        print("Limited GPU memory. A smaller generator may be safer.")

    else:

        print("\nGPU Memory Assessment:")
        print("GPU memory is too limited for an 8B 4-bit model.")

else:

    print("CUDA Available : False")
    print("\nNo GPU is currently available.")
    print("Do not attempt to load an 8B model in this runtime.")


print("=" * 70)

COLAB GPU RESOURCE CHECK
CUDA Available : True
GPU Name      : Tesla T4
Total VRAM    : 14.56 GB
Allocated VRAM: 1.66 GB
Reserved VRAM : 1.75 GB

GPU Memory Assessment:
Sufficient for attempting an 8B 4-bit model.


In [16]:
# ============================================================
# STEP 16: LOAD CPU-COMPATIBLE GENERATION MODEL
# ============================================================
# The current Colab runtime does not have a GPU.
#
# Therefore, Llama 3.1 8B is NOT loaded in this runtime.
#
# For today's working prototype, we use:
#
#     google/flan-t5-base
#
# This is a smaller instruction-tuned model that can run on
# CPU and allows us to complete and demonstrate the RAG
# pipeline without waiting for GPU access or Llama approval.
#
# IMPORTANT:
# The final report must identify the actual generator model
# used in the experiment. We will NOT claim that Llama 3.1 8B
# was executed if it was not.
# ============================================================


# ------------------------------------------------------------
# IMPORT TRANSFORMERS COMPONENTS
# ------------------------------------------------------------

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)


# ------------------------------------------------------------
# DEFINE GENERATION MODEL
# ------------------------------------------------------------

GENERATOR_MODEL_ID = "google/flan-t5-base"


print("=" * 70)
print("LOADING GENERATION MODEL")
print("=" * 70)

print(f"Model: {GENERATOR_MODEL_ID}")
print("Device: CPU")


# ------------------------------------------------------------
# LOAD TOKENIZER
# ------------------------------------------------------------

generator_tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL_ID
)


# ------------------------------------------------------------
# LOAD MODEL
# ------------------------------------------------------------

generator_model = AutoModelForSeq2SeqLM.from_pretrained(
    GENERATOR_MODEL_ID
)


# ------------------------------------------------------------
# Force CPU execution
# ------------------------------------------------------------

generator_model = generator_model.to("cpu")


# ------------------------------------------------------------
# Put model into evaluation mode
# ------------------------------------------------------------

generator_model.eval()


print("\n")
print("=" * 70)
print("GENERATION MODEL LOADED SUCCESSFULLY")
print("=" * 70)

print(f"Model : {GENERATOR_MODEL_ID}")
print("Device: CPU")

print("=" * 70)

LOADING GENERATION MODEL
Model: google/flan-t5-base
Device: CPU


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]



GENERATION MODEL LOADED SUCCESSFULLY
Model : google/flan-t5-base
Device: CPU


In [17]:
# ============================================================
# STEP 17: CREATE COMPLETE RAG ANSWER FUNCTION
# ============================================================
# This function combines:
#
#   1. Semantic retrieval
#   2. NHS context construction
#   3. Prompt construction
#   4. LLM generation
#
# The function returns both the generated answer and the
# retrieved NHS sources so that the eventual web interface
# can display the evidence used to generate the answer.
# ============================================================


def generate_rag_answer(
    question,
    embedding_model_name="BGE",
    top_k=5,
    max_new_tokens=180
):

    # --------------------------------------------------------
    # STEP 1: RETRIEVE NHS DOCUMENTS
    # --------------------------------------------------------

    retrieval_start = time.time()

    retrieved_results = retrieve_documents(
        question=question,
        model_name=embedding_model_name,
        top_k=top_k
    )

    retrieval_time = (
        time.time() - retrieval_start
    )


    # --------------------------------------------------------
    # STEP 2: BUILD NHS CONTEXT
    # --------------------------------------------------------

    context = build_context(
        retrieved_results
    )


    # --------------------------------------------------------
    # STEP 3: CREATE RAG PROMPT
    # --------------------------------------------------------

    prompt = create_rag_prompt(
        question=question,
        context=context
    )


    # --------------------------------------------------------
    # STEP 4: TOKENIZE PROMPT
    # --------------------------------------------------------

    inputs = generator_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )


    # --------------------------------------------------------
    # STEP 5: GENERATE ANSWER
    # --------------------------------------------------------

    generation_start = time.time()

    with torch.no_grad():

        output_ids = generator_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=2
        )

    generation_time = (
        time.time() - generation_start
    )


    # --------------------------------------------------------
    # STEP 6: DECODE GENERATED ANSWER
    # --------------------------------------------------------

    answer = generator_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()


    # --------------------------------------------------------
    # STEP 7: RETURN COMPLETE RESULT
    # --------------------------------------------------------

    return {

        "question": question,

        "answer": answer,

        "retrieved_documents": retrieved_results,

        "retrieval_time": retrieval_time,

        "generation_time": generation_time,

        "embedding_model": embedding_model_name,

        "top_k": top_k

    }

In [18]:
# ============================================================
# STEP 18: FIRST COMPLETE RAG TEST
# ============================================================
# This cell tests the complete RAG pipeline:
#
#   User Question
#        ↓
#   BGE Query Embedding
#        ↓
#   FAISS Retrieval
#        ↓
#   NHS Context
#        ↓
#   RAG Prompt
#        ↓
#   FLAN-T5
#        ↓
#   Generated Answer
#
# We also display the retrieved NHS sources and performance
# timings.
# ============================================================


# ------------------------------------------------------------
# TEST QUESTION
# ------------------------------------------------------------

test_question = (
    "What are the symptoms of asthma?"
)


print("=" * 70)
print("COMPLETE RAG SYSTEM TEST")
print("=" * 70)

print(f"Question: {test_question}")


# ------------------------------------------------------------
# RUN COMPLETE RAG PIPELINE
# ------------------------------------------------------------

rag_result = generate_rag_answer(
    question=test_question,
    embedding_model_name="BGE",
    top_k=5,
    max_new_tokens=180
)


# ------------------------------------------------------------
# DISPLAY GENERATED ANSWER
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("GENERATED ANSWER")
print("=" * 70)

print(rag_result["answer"])


# ------------------------------------------------------------
# DISPLAY RETRIEVED SOURCES
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("RETRIEVED NHS SOURCES")
print("=" * 70)

for result in rag_result["retrieved_documents"]:

    print(
        f"Rank {result['rank']} | "
        f"Score {result['score']:.4f} | "
        f"{result['source_file']}"
    )


# ------------------------------------------------------------
# DISPLAY PERFORMANCE
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("RAG PERFORMANCE")
print("=" * 70)

print(
    f"Retrieval Time : "
    f"{rag_result['retrieval_time']:.4f} seconds"
)

print(
    f"Generation Time: "
    f"{rag_result['generation_time']:.4f} seconds"
)

print(
    f"Total Time     : "
    f"{rag_result['retrieval_time'] + rag_result['generation_time']:.4f} seconds"
)

print("=" * 70)

COMPLETE RAG SYSTEM TEST
Question: What are the symptoms of asthma?


GENERATED ANSWER
• wheezing • coughing • shortness of breath • your chest feeling tight Asthma symptoms can be mild or severe. They usually come and go. They may be worse at night and early in the morning. An asthma attack is when you have severe symptoms and it's difficult to breathe, which can be life-threatening. Symptoms can be triggered by different things, such as allergies, exercise, smoke, cold air or infections. It's not fully understood what causes asthma. You may be more likely to get asthma if: • you have allergies such as eczema, hay fever or food allergies • other people in your family have asthma, hay fever, eczema or allergies • you were born prematurely or had a low birth weight • you were exposed to things like high levels of air


RETRIEVED NHS SOURCES
Rank 1 | Score 0.8265 | asthma.txt
Rank 2 | Score 0.8027 | asthma.txt
Rank 3 | Score 0.7476 | asthma.txt
Rank 4 | Score 0.7467 | hives.txt
Rank 5 | 

In [19]:
# ============================================================
# STEP 19: VALIDATE THE COMPLETE RAG RESPONSE
# ============================================================
# This cell checks the output of the complete RAG pipeline.
#
# We verify:
#   1. A non-empty answer was generated.
#   2. NHS documents were retrieved.
#   3. Similarity scores are available.
#   4. Retrieval and generation timings were recorded.
#
# This is a technical pipeline check.
# It does NOT determine medical correctness automatically.
# Medical/content evaluation will be handled separately.
# ============================================================


print("=" * 70)
print("RAG RESPONSE VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# CHECK 1: GENERATED ANSWER
# ------------------------------------------------------------

answer = rag_result["answer"]

if answer and len(answer.strip()) > 0:

    print("Generated Answer : PASS")

else:

    print("Generated Answer : FAIL")


# ------------------------------------------------------------
# CHECK 2: RETRIEVED DOCUMENTS
# ------------------------------------------------------------

documents = rag_result[
    "retrieved_documents"
]

if len(documents) > 0:

    print(
        f"Retrieved Documents : PASS "
        f"({len(documents)})"
    )

else:

    print("Retrieved Documents : FAIL")


# ------------------------------------------------------------
# CHECK 3: SIMILARITY SCORES
# ------------------------------------------------------------

valid_scores = all(
    "score" in document
    for document in documents
)

if valid_scores:

    print("Similarity Scores : PASS")

else:

    print("Similarity Scores : FAIL")


# ------------------------------------------------------------
# CHECK 4: RETRIEVAL TIMING
# ------------------------------------------------------------

if rag_result["retrieval_time"] >= 0:

    print("Retrieval Timing : PASS")

else:

    print("Retrieval Timing : FAIL")


# ------------------------------------------------------------
# CHECK 5: GENERATION TIMING
# ------------------------------------------------------------

if rag_result["generation_time"] >= 0:

    print("Generation Timing : PASS")

else:

    print("Generation Timing : FAIL")


# ------------------------------------------------------------
# DISPLAY FINAL ANSWER
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("ANSWER TO VALIDATE")
print("=" * 70)

print(answer)


print("\n")
print("=" * 70)
print("RAG RESPONSE VALIDATION COMPLETED")
print("=" * 70)

RAG RESPONSE VALIDATION
Generated Answer : PASS
Retrieved Documents : PASS (5)
Similarity Scores : PASS
Retrieval Timing : PASS
Generation Timing : PASS


ANSWER TO VALIDATE
• wheezing • coughing • shortness of breath • your chest feeling tight Asthma symptoms can be mild or severe. They usually come and go. They may be worse at night and early in the morning. An asthma attack is when you have severe symptoms and it's difficult to breathe, which can be life-threatening. Symptoms can be triggered by different things, such as allergies, exercise, smoke, cold air or infections. It's not fully understood what causes asthma. You may be more likely to get asthma if: • you have allergies such as eczema, hay fever or food allergies • other people in your family have asthma, hay fever, eczema or allergies • you were born prematurely or had a low birth weight • you were exposed to things like high levels of air


RAG RESPONSE VALIDATION COMPLETED


In [20]:
# ============================================================
# STEP 20A: LOAD LLAMA 3.1 8B INSTRUCT
# ============================================================
# This replaces the previous FLAN-T5 generation model.
#
# Model:
#   meta-llama/Llama-3.1-8B-Instruct
#
# Hardware:
#   Google Colab T4 GPU
#
# Quantization:
#   4-bit NF4
#
# ============================================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from huggingface_hub import login


# ============================================================
# 1. VERIFY GPU
# ============================================================

print("=" * 70)
print("GPU CHECK")
print("=" * 70)

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU is not available. "
        "Please select T4 GPU in Runtime > Change runtime type."
    )

print(
    f"GPU: {torch.cuda.get_device_name(0)}"
)

print(
    f"VRAM: "
    f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB"
)


# ============================================================
# 2. HUGGING FACE LOGIN
# ============================================================

print("\n")
print("=" * 70)
print("HUGGING FACE LOGIN")
print("=" * 70)

login()

print("Hugging Face authentication completed.")


# ============================================================
# 3. DEFINE MODEL
# ============================================================

LLAMA_MODEL_ID = (
    "meta-llama/Llama-3.1-8B-Instruct"
)


# ============================================================
# 4. DEFINE 4-BIT QUANTIZATION
# ============================================================

quantization_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)


# ============================================================
# 5. LOAD TOKENIZER
# ============================================================

print("\nLoading Llama tokenizer...")

llama_tokenizer = AutoTokenizer.from_pretrained(
    LLAMA_MODEL_ID
)


# ============================================================
# 6. LOAD LLAMA
# ============================================================

print("\nLoading Llama 3.1 8B Instruct...")
print("This can take several minutes.")

llama_model = AutoModelForCausalLM.from_pretrained(

    LLAMA_MODEL_ID,

    quantization_config=quantization_config,

    device_map="auto"
)


# ============================================================
# 7. CONFIGURE TOKENIZER
# ============================================================

if llama_tokenizer.pad_token is None:

    llama_tokenizer.pad_token = (
        llama_tokenizer.eos_token
    )


# ============================================================
# 8. EVALUATION MODE
# ============================================================

llama_model.eval()


# ============================================================
# 9. DISPLAY MODEL STATUS
# ============================================================

print("\n")
print("=" * 70)
print("LLAMA 3.1 8B INSTRUCT LOADED SUCCESSFULLY")
print("=" * 70)

print(
    f"Model : {LLAMA_MODEL_ID}"
)

print(
    f"Device: {llama_model.device}"
)

print("=" * 70)

GPU CHECK
GPU: Tesla T4
VRAM: 14.56 GB


HUGGING FACE LOGIN


Hugging Face authentication completed.

Loading Llama tokenizer...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]



LLAMA 3.1 8B INSTRUCT LOADED SUCCESSFULLY
Model : meta-llama/Llama-3.1-8B-Instruct
Device: cuda:0


In [21]:
# ============================================================
# STEP 20B: TEST LLAMA 3.1 8B INSTRUCT
# ============================================================
# This is an isolated generation test.
#
# We are NOT using FAISS or NHS retrieval yet.
#
# Purpose:
#   Confirm that:
#   1. The model loaded correctly.
#   2. The tokenizer works.
#   3. The T4 GPU can perform generation.
#   4. The chat template works.
# ============================================================

test_messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful healthcare information assistant. "
            "Give clear and concise informational answers."
        )
    },
    {
        "role": "user",
        "content": "What are the symptoms of asthma?"
    }
]


# ------------------------------------------------------------
# CREATE CHAT-TEMPLATE INPUT
# ------------------------------------------------------------

prompt = llama_tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True
)


# ------------------------------------------------------------
# TOKENIZE
# ------------------------------------------------------------

inputs = llama_tokenizer(
    prompt,
    return_tensors="pt"
)


# ------------------------------------------------------------
# MOVE INPUTS TO GPU
# ------------------------------------------------------------

inputs = {
    key: value.to("cuda")
    for key, value in inputs.items()
}


# ------------------------------------------------------------
# GENERATE
# ------------------------------------------------------------

print("=" * 70)
print("LLAMA GENERATION TEST")
print("=" * 70)

print("Generating answer...")

with torch.no_grad():

    output_ids = llama_model.generate(

        **inputs,

        max_new_tokens=150,

        do_sample=False,

        temperature=None,

        pad_token_id=llama_tokenizer.eos_token_id
    )


# ------------------------------------------------------------
# REMOVE INPUT TOKENS
# ------------------------------------------------------------

generated_ids = output_ids[
    0,
    inputs["input_ids"].shape[1]:
]


# ------------------------------------------------------------
# DECODE
# ------------------------------------------------------------

generated_answer = llama_tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("LLAMA TEST ANSWER")
print("=" * 70)

print(generated_answer)

print("\n")
print("=" * 70)
print("LLAMA GENERATION TEST COMPLETED")
print("=" * 70)

LLAMA GENERATION TEST
Generating answer...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.




LLAMA TEST ANSWER
The symptoms of asthma can vary from person to person, but common symptoms include:

1.  **Wheezing**: A high-pitched whistling sound when breathing out, usually due to narrowed airways.
2.  **Coughing**: A persistent cough, often worse at night or after exercise.
3.  **Shortness of breath**: Feeling like you can't catch your breath or feeling winded even when doing light activities.
4.  **Chest tightness**: Feeling like your chest is constricted or tight.
5.  **Difficulty breathing**: Feeling like you're having trouble inhaling or exhaling.
6.  **Recurring coughing fits**: Coughing spells that can be triggered by specific factors


LLAMA GENERATION TEST COMPLETED


In [22]:
# ============================================================
# STEP 21: CONNECT LLAMA TO THE RAG PIPELINE
# ============================================================
# This function replaces the previous FLAN-T5 generation
# function.
#
# Complete pipeline:
#
#   Question
#      ↓
#   Embedding Model
#      ↓
#   FAISS Retrieval
#      ↓
#   NHS Context
#      ↓
#   Llama 3.1 8B Instruct
#      ↓
#   Generated Answer
# ============================================================


def generate_llama_rag_answer(
    question,
    embedding_model_name="BGE",
    top_k=5,
    max_new_tokens=180
):

    # ========================================================
    # STEP 1: RETRIEVE NHS DOCUMENTS
    # ========================================================

    retrieval_start = time.time()

    retrieved_results = retrieve_documents(
        question=question,
        model_name=embedding_model_name,
        top_k=top_k
    )

    retrieval_time = (
        time.time() - retrieval_start
    )


    # ========================================================
    # STEP 2: BUILD NHS CONTEXT
    # ========================================================

    context = build_context(
        retrieved_results
    )


    # ========================================================
    # STEP 3: CREATE SYSTEM INSTRUCTION
    # ========================================================

    system_message = """
You are a healthcare information assistant.

Your answers must be based only on the NHS information
provided in the retrieved context.

Rules:
1. Use the retrieved NHS context as the primary source.
2. Do not invent medical facts.
3. Do not diagnose the user.
4. Do not prescribe treatment.
5. If the retrieved context does not contain enough
   information to answer the question, say so clearly.
6. Give a concise and clear answer.
7. Preserve important NHS safety or emergency guidance
   contained in the retrieved context.
"""


    # ========================================================
    # STEP 4: CREATE USER MESSAGE
    # ========================================================

    user_message = f"""
Answer the following healthcare question using the
retrieved NHS information.

QUESTION:
{question}

RETRIEVED NHS CONTEXT:
{context}
"""


    # ========================================================
    # STEP 5: CREATE LLAMA CHAT INPUT
    # ========================================================

    messages = [

        {
            "role": "system",
            "content": system_message
        },

        {
            "role": "user",
            "content": user_message
        }

    ]


    prompt = llama_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    # ========================================================
    # STEP 6: TOKENIZE
    # ========================================================

    inputs = llama_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )


    # ========================================================
    # STEP 7: MOVE INPUTS TO GPU
    # ========================================================

    inputs = {
        key: value.to("cuda")
        for key, value in inputs.items()
    }


    # ========================================================
    # STEP 8: GENERATE ANSWER
    # ========================================================

    generation_start = time.time()

    with torch.no_grad():

        output_ids = llama_model.generate(

            **inputs,

            max_new_tokens=max_new_tokens,

            do_sample=False,

            pad_token_id=(
                llama_tokenizer.eos_token_id
            )
        )


    generation_time = (
        time.time() - generation_start
    )


    # ========================================================
    # STEP 9: REMOVE PROMPT TOKENS
    # ========================================================

    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[1]:
    ]


    # ========================================================
    # STEP 10: DECODE ANSWER
    # ========================================================

    answer = llama_tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()


    # ========================================================
    # STEP 11: RETURN COMPLETE RESULT
    # ========================================================

    return {

        "question": question,

        "answer": answer,

        "retrieved_documents": retrieved_results,

        "retrieval_time": retrieval_time,

        "generation_time": generation_time,

        "total_time": (
            retrieval_time +
            generation_time
        ),

        "embedding_model": embedding_model_name,

        "top_k": top_k,

        "generator_model": (
            "meta-llama/"
            "Llama-3.1-8B-Instruct"
        )

    }


print("=" * 70)
print("LLAMA RAG FUNCTION CREATED")
print("=" * 70)

print("Retriever        : FAISS")
print("Knowledge Source : NHS")
print("Generator        : Llama 3.1 8B Instruct")
print("Quantization     : 4-bit")
print("Default Top-K    : 5")

print("=" * 70)

LLAMA RAG FUNCTION CREATED
Retriever        : FAISS
Knowledge Source : NHS
Generator        : Llama 3.1 8B Instruct
Quantization     : 4-bit
Default Top-K    : 5


In [23]:
# ============================================================
# STEP 22: COMPLETE LLAMA RAG TEST
# ============================================================
# Pipeline:
#
# Question
#    ↓
# BGE
#    ↓
# FAISS
#    ↓
# NHS Context
#    ↓
# Llama 3.1 8B Instruct
#    ↓
# Answer
# ============================================================

test_question = "What are the symptoms of asthma?"

print("=" * 70)
print("COMPLETE LLAMA RAG TEST")
print("=" * 70)

print(f"Question: {test_question}")
print("Embedding Model: BGE")
print("Top-K: 5")
print("Generator: Llama 3.1 8B Instruct")
print("\nGenerating answer...")


# ------------------------------------------------------------
# RUN COMPLETE RAG PIPELINE
# ------------------------------------------------------------

llama_rag_result = generate_llama_rag_answer(
    question=test_question,
    embedding_model_name="BGE",
    top_k=5,
    max_new_tokens=180
)


# ------------------------------------------------------------
# DISPLAY ANSWER
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("GENERATED ANSWER")
print("=" * 70)

print(llama_rag_result["answer"])


# ------------------------------------------------------------
# DISPLAY RETRIEVED SOURCES
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("RETRIEVED NHS SOURCES")
print("=" * 70)

for result in llama_rag_result["retrieved_documents"]:

    print(
        f"Rank {result['rank']} | "
        f"Similarity: {result['score']:.4f} | "
        f"Source: {result['source_file']} | "
        f"Chunk: {result['chunk_id']}"
    )


# ------------------------------------------------------------
# DISPLAY PERFORMANCE
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("PERFORMANCE")
print("=" * 70)

print(
    f"Retrieval Time : "
    f"{llama_rag_result['retrieval_time']:.4f} seconds"
)

print(
    f"Generation Time: "
    f"{llama_rag_result['generation_time']:.4f} seconds"
)

print(
    f"Total Time     : "
    f"{llama_rag_result['total_time']:.4f} seconds"
)

print("=" * 70)

COMPLETE LLAMA RAG TEST
Question: What are the symptoms of asthma?
Embedding Model: BGE
Top-K: 5
Generator: Llama 3.1 8B Instruct

Generating answer...


GENERATED ANSWER
Based on the retrieved NHS context, the symptoms of asthma are:

• wheezing
• coughing
• shortness of breath
• your chest feeling tight

These symptoms can be mild or severe and may be triggered by different things, such as exercise, high levels of air pollution, cold air, or contact with something you're allergic to, such as pollen, dust, mould, or animals.


RETRIEVED NHS SOURCES
Rank 1 | Similarity: 0.8265 | Source: asthma.txt | Chunk: 1
Rank 2 | Similarity: 0.8027 | Source: asthma.txt | Chunk: 12
Rank 3 | Similarity: 0.7476 | Source: asthma.txt | Chunk: 10
Rank 4 | Similarity: 0.7467 | Source: hives.txt | Chunk: 13
Rank 5 | Similarity: 0.7467 | Source: food_allergy.txt | Chunk: 2


PERFORMANCE
Retrieval Time : 0.0167 seconds
Generation Time: 8.2493 seconds
Total Time     : 8.2660 seconds


In [24]:
# ============================================================
# STEP 23: CREATE LLAMA CHATBOT BACKEND
# ============================================================
# This function is the backend that the web interface will use.
#
# It returns:
#   - Generated answer
#   - Retrieved NHS sources
#   - Similarity scores
#   - Retrieval time
#   - Generation time
#   - Total response time
# ============================================================


def llama_chatbot_response(
    question,
    embedding_model_name="BGE",
    top_k=5
):

    # --------------------------------------------------------
    # Validate question
    # --------------------------------------------------------

    if not question or not question.strip():

        return {
            "answer": "Please enter a healthcare question.",
            "sources": [],
            "retrieval_time": 0.0,
            "generation_time": 0.0,
            "total_time": 0.0
        }


    # --------------------------------------------------------
    # Run Llama RAG pipeline
    # --------------------------------------------------------

    result = generate_llama_rag_answer(

        question=question.strip(),

        embedding_model_name=embedding_model_name,

        top_k=int(top_k),

        max_new_tokens=180
    )


    # --------------------------------------------------------
    # Prepare source information
    # --------------------------------------------------------

    sources = []

    for document in result["retrieved_documents"]:

        sources.append({

            "rank": document["rank"],

            "source_file": document["source_file"],

            "chunk_id": document["chunk_id"],

            "similarity": document["score"]

        })


    # --------------------------------------------------------
    # Return clean response
    # --------------------------------------------------------

    return {

        "answer": result["answer"],

        "sources": sources,

        "retrieval_time": result["retrieval_time"],

        "generation_time": result["generation_time"],

        "total_time": result["total_time"]

    }


print("=" * 70)
print("LLAMA CHATBOT BACKEND CREATED")
print("=" * 70)

print("Generator: Llama 3.1 8B Instruct")
print("Retriever: FAISS")
print("Knowledge Base: NHS")

print("=" * 70)

LLAMA CHATBOT BACKEND CREATED
Generator: Llama 3.1 8B Instruct
Retriever: FAISS
Knowledge Base: NHS


In [25]:
# ============================================================
# STEP 24: TEST LLAMA CHATBOT BACKEND
# ============================================================

test_questions = [
    "What are the symptoms of asthma?",
    "What is aciclovir used to treat?",
    "What are the symptoms of a food allergy?"
]

print("=" * 70)
print("LLAMA CHATBOT BACKEND TEST")
print("=" * 70)

for number, question in enumerate(
    test_questions,
    start=1
):

    print("\n")
    print("-" * 70)
    print(f"TEST {number}")
    print("-" * 70)

    print(f"Question: {question}")
    print("Generating...")

    response = llama_chatbot_response(
        question=question,
        embedding_model_name="BGE",
        top_k=5
    )

    print("\nAnswer:")
    print(response["answer"])

    print("\nRetrieved Sources:")

    for source in response["sources"]:

        print(
            f"Rank {source['rank']} | "
            f"{source['source_file']} | "
            f"Chunk {source['chunk_id']} | "
            f"Similarity {source['similarity']:.4f}"
        )

    print("\nPerformance:")

    print(
        f"Retrieval : "
        f"{response['retrieval_time']:.4f} seconds"
    )

    print(
        f"Generation: "
        f"{response['generation_time']:.4f} seconds"
    )

    print(
        f"Total     : "
        f"{response['total_time']:.4f} seconds"
    )


print("\n")
print("=" * 70)
print("LLAMA CHATBOT BACKEND TEST COMPLETED")
print("=" * 70)

LLAMA CHATBOT BACKEND TEST


----------------------------------------------------------------------
TEST 1
----------------------------------------------------------------------
Question: What are the symptoms of asthma?
Generating...

Answer:
Based on the retrieved NHS context, the symptoms of asthma are:

• wheezing
• coughing
• shortness of breath
• your chest feeling tight

These symptoms can be mild or severe and may be triggered by different things, such as exercise, high levels of air pollution, cold air, or contact with something you're allergic to, such as pollen, dust, mould, or animals.

Retrieved Sources:
Rank 1 | asthma.txt | Chunk 1 | Similarity 0.8265
Rank 2 | asthma.txt | Chunk 12 | Similarity 0.8027
Rank 3 | asthma.txt | Chunk 10 | Similarity 0.7476
Rank 4 | hives.txt | Chunk 13 | Similarity 0.7467
Rank 5 | food_allergy.txt | Chunk 2 | Similarity 0.7467

Performance:
Retrieval : 0.0169 seconds
Generation: 7.9093 seconds
Total     : 7.9262 seconds


--------------------

In [26]:
# ============================================================
# STEP 25: SAVE LLAMA RAG TEST RESULTS
# ============================================================
# Saves the Llama RAG test outputs for reproducibility and
# dissertation evidence.
# ============================================================

import json
from pathlib import Path


# ------------------------------------------------------------
# RESULTS DIRECTORY
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Healthcare_RAG"
)

RESULTS_DIR = PROJECT_DIR / "results"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# QUESTIONS
# ------------------------------------------------------------

test_questions = [
    "What are the symptoms of asthma?",
    "What is aciclovir used to treat?",
    "What are the symptoms of a food allergy?"
]


# ------------------------------------------------------------
# RUN AND SAVE RESULTS
# ------------------------------------------------------------

llama_test_results = []

for question in test_questions:

    print(f"Testing: {question}")

    response = llama_chatbot_response(
        question=question,
        embedding_model_name="BGE",
        top_k=5
    )

    llama_test_results.append({

        "question": question,

        "answer": response["answer"],

        "sources": response["sources"],

        "retrieval_time_seconds": (
            response["retrieval_time"]
        ),

        "generation_time_seconds": (
            response["generation_time"]
        ),

        "total_time_seconds": (
            response["total_time"]
        ),

        "embedding_model": "BGE",

        "generator_model":
            "meta-llama/Llama-3.1-8B-Instruct",

        "top_k": 5

    })


# ------------------------------------------------------------
# SAVE JSON
# ------------------------------------------------------------

llama_results_path = (
    RESULTS_DIR /
    "llama_rag_test_results.json"
)

with open(
    llama_results_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        llama_test_results,
        file,
        ensure_ascii=False,
        indent=4
    )


# ------------------------------------------------------------
# CONFIRMATION
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("LLAMA RAG TEST RESULTS SAVED")
print("=" * 70)

print(
    f"Test Cases : {len(llama_test_results)}"
)

print(
    f"File       : {llama_results_path}"
)

print("=" * 70)

Testing: What are the symptoms of asthma?
Testing: What is aciclovir used to treat?
Testing: What are the symptoms of a food allergy?


LLAMA RAG TEST RESULTS SAVED
Test Cases : 3
File       : /content/drive/MyDrive/Healthcare_RAG/results/llama_rag_test_results.json


In [28]:
# ============================================================
# STEP 26: FINAL LLAMA RAG WEB INTERFACE
# ============================================================
# User-facing application:
#
#   User Question
#        ↓
#   Selected Embedding Model
#        ↓
#   FAISS Retrieval
#        ↓
#   NHS Context
#        ↓
#   Llama 3.1 8B Instruct
#        ↓
#   Answer + Sources + Performance
#
# This is the final interface for the prototype.
# ============================================================

import gradio as gr


# ============================================================
# CHATBOT HANDLER
# ============================================================

def run_llama_interface(
    question,
    embedding_model,
    top_k
):

    # --------------------------------------------------------
    # Validate question
    # --------------------------------------------------------

    if not question or not question.strip():

        return (
            "Please enter a healthcare question.",
            "No sources retrieved.",
            "No query was submitted."
        )


    # --------------------------------------------------------
    # Run Llama RAG
    # --------------------------------------------------------

    response = llama_chatbot_response(

        question=question.strip(),

        embedding_model_name=embedding_model,

        top_k=int(top_k)
    )


    # --------------------------------------------------------
    # Format retrieved sources
    # --------------------------------------------------------

    source_lines = []

    for source in response["sources"]:

        source_lines.append(
            f"{source['rank']}. "
            f"{source['source_file']} | "
            f"Chunk {source['chunk_id']} | "
            f"Similarity: "
            f"{source['similarity']:.4f}"
        )


    source_text = "\n".join(
        source_lines
    )


    # --------------------------------------------------------
    # Format performance
    # --------------------------------------------------------

    performance_text = (
        f"Generator: Llama 3.1 8B Instruct\n"
        f"Embedding Model: {embedding_model}\n"
        f"Top-K: {int(top_k)}\n"
        f"Retrieval Time: "
        f"{response['retrieval_time']:.4f} seconds\n"
        f"Generation Time: "
        f"{response['generation_time']:.4f} seconds\n"
        f"Total Response Time: "
        f"{response['total_time']:.4f} seconds"
    )


    return (
        response["answer"],
        source_text,
        performance_text
    )


# ============================================================
# CLEAR FUNCTION
# ============================================================

def clear_llama_interface():

    return (
        "",
        "",
        "",
        "BGE",
        5
    )


# ============================================================
# BUILD INTERFACE
# ============================================================

with gr.Blocks(
    title="NHS Healthcare RAG Assistant"
) as llama_demo:

    gr.Markdown(
        """
        # NHS Healthcare RAG Assistant

        This research prototype retrieves relevant information
        from an NHS knowledge base and generates an answer using
        the retrieved evidence.
        """
    )


    # --------------------------------------------------------
    # Question
    # --------------------------------------------------------

    question_input = gr.Textbox(

        label="Healthcare Question",

        placeholder=(
            "Example: What are the symptoms of asthma?"
        ),

        lines=4
    )


    # --------------------------------------------------------
    # Controls
    # --------------------------------------------------------

    with gr.Row():

        embedding_input = gr.Dropdown(

            choices=MODEL_NAMES,

            value="BGE",

            label="Embedding Model"
        )


        top_k_input = gr.Slider(

            minimum=1,

            maximum=10,

            value=5,

            step=1,

            label="Retrieved NHS Chunks"
        )


    # --------------------------------------------------------
    # Buttons
    # --------------------------------------------------------

    with gr.Row():

        ask_button = gr.Button(
            "Ask NHS Assistant"
        )

        clear_button = gr.Button(
            "Clear"
        )


    # --------------------------------------------------------
    # Answer
    # --------------------------------------------------------

    answer_output = gr.Textbox(

        label="Generated Answer",

        lines=10
    )


    # --------------------------------------------------------
    # Sources
    # --------------------------------------------------------

    sources_output = gr.Textbox(

        label="Retrieved NHS Sources",

        lines=8
    )


    # --------------------------------------------------------
    # Performance
    # --------------------------------------------------------

    performance_output = gr.Textbox(

        label="System Performance",

        lines=6
    )


    # ========================================================
# ASK BUTTON
# ========================================================

ask_button.click(

    fn=run_llama_interface,

    inputs=[
        question_input,
        embedding_input,
        top_k_input
    ],

    outputs=[
        answer_output,
        sources_output,
        performance_output
    ]
)


    # ========================================================
    # CLEAR BUTTON
    # ========================================================

    clear_button.click(

        fn=clear_llama_interface,

        inputs=[],

        outputs=[
            question_input,
            answer_output,
            sources_output,
            performance_output,
            embedding_input,
            top_k_input
        ]
    )


# ============================================================
# LAUNCH
# ============================================================

llama_demo.launch(
    share=True
)

IndentationError: unexpected indent (2765532460.py, line 264)

In [ ]:
# ============================================================
# STEP 27: FINAL LLAMA RAG SYSTEM VERIFICATION
# ============================================================
# Verifies that the complete Llama-based RAG system is ready:
#
#   FAISS
#   NHS metadata
#   Query embedding models
#   Llama 3.1 8B Instruct
#   Retrieval
#   RAG generation
#   Saved test results
# ============================================================

from pathlib import Path
import json


# ------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Healthcare_RAG"
)

FAISS_DIR = PROJECT_DIR / "data" / "faiss"

RESULTS_DIR = PROJECT_DIR / "results"


MODEL_NAMES = [
    "MiniLM",
    "MPNet",
    "E5",
    "BGE"
]


all_checks_passed = True


print("=" * 70)
print("FINAL LLAMA RAG SYSTEM VERIFICATION")
print("=" * 70)


# ============================================================
# 1. FAISS CHECK
# ============================================================

print("\nFAISS INDEXES")
print("-" * 70)

for model_name in MODEL_NAMES:

    index_path = (
        FAISS_DIR /
        model_name /
        "faiss_index.bin"
    )

    metadata_path = (
        FAISS_DIR /
        model_name /
        "metadata.json"
    )

    index_ok = index_path.exists()
    metadata_ok = metadata_path.exists()

    print(
        f"{model_name:10} | "
        f"Index: {'PASS' if index_ok else 'FAIL'} | "
        f"Metadata: {'PASS' if metadata_ok else 'FAIL'}"
    )

    if not index_ok or not metadata_ok:

        all_checks_passed = False


# ============================================================
# 2. QUERY MODELS
# ============================================================

print("\nQUERY EMBEDDING MODELS")
print("-" * 70)

if "query_models" in globals():

    for model_name in MODEL_NAMES:

        passed = model_name in query_models

        print(
            f"{model_name:10} : "
            f"{'PASS' if passed else 'FAIL'}"
        )

        if not passed:

            all_checks_passed = False

else:

    print("query_models : FAIL")

    all_checks_passed = False


# ============================================================
# 3. LLAMA MODEL
# ============================================================

print("\nGENERATION MODEL")
print("-" * 70)

if "llama_model" in globals():

    print(
        "Llama 3.1 8B Instruct : PASS"
    )

else:

    print(
        "Llama 3.1 8B Instruct : FAIL"
    )

    all_checks_passed = False


# ============================================================
# 4. LLAMA TOKENIZER
# ============================================================

if "llama_tokenizer" in globals():

    print(
        "Llama tokenizer       : PASS"
    )

else:

    print(
        "Llama tokenizer       : FAIL"
    )

    all_checks_passed = False


# ============================================================
# 5. RAG FUNCTIONS
# ============================================================

print("\nRAG FUNCTIONS")
print("-" * 70)

required_functions = [

    "retrieve_documents",
    "build_context",
    "generate_llama_rag_answer",
    "llama_chatbot_response"

]

for function_name in required_functions:

    passed = function_name in globals()

    print(
        f"{function_name:30} : "
        f"{'PASS' if passed else 'FAIL'}"
    )

    if not passed:

        all_checks_passed = False


# ============================================================
# 6. SAVED LLAMA TEST RESULTS
# ============================================================

print("\nSAVED TEST RESULTS")
print("-" * 70)

llama_results_path = (
    RESULTS_DIR /
    "llama_rag_test_results.json"
)

if llama_results_path.exists():

    with open(
        llama_results_path,
        "r",
        encoding="utf-8"
    ) as file:

        llama_results = json.load(file)

    print(
        f"Llama test cases : {len(llama_results)}"
    )

    print(
        "Saved results     : PASS"
    )

else:

    print(
        "Saved results     : FAIL"
    )

    all_checks_passed = False


# ============================================================
# 7. FINAL RESULT
# ============================================================

print("\n")
print("=" * 70)

if all_checks_passed:

    print("LLAMA RAG SYSTEM VERIFICATION PASSED")

else:

    print("LLAMA RAG SYSTEM VERIFICATION FAILED")

print("=" * 70)

In [ ]:
# ============================================================
# STEP 28: CREATE FORMAL RETRIEVAL EVALUATION DATASET
# ============================================================
# Purpose:
#   Create a controlled evaluation set for comparing:
#
#       MiniLM
#       MPNet
#       E5
#       BGE
#
# Metrics to calculate later:
#
#       Precision@K
#       Recall@K
#       MRR
#       nDCG
#
# IMPORTANT:
#   These questions are paired with the NHS source article
#   that should contain the relevant information.
#
#   The source filenames must match the actual cleaned dataset.
# ============================================================

evaluation_questions = [

    {
        "id": 1,
        "question": "What are the symptoms of asthma?",
        "relevant_sources": ["asthma.txt"]
    },

    {
        "id": 2,
        "question": "What are the causes of asthma?",
        "relevant_sources": ["asthma.txt"]
    },

    {
        "id": 3,
        "question": "How is asthma diagnosed?",
        "relevant_sources": ["asthma.txt"]
    },

    {
        "id": 4,
        "question": "What is aciclovir used to treat?",
        "relevant_sources": [
            "aciclovir_zovirax.txt"
        ]
    },

    {
        "id": 5,
        "question": "What are the symptoms of a food allergy?",
        "relevant_sources": [
            "food_allergy.txt"
        ]
    },

    {
        "id": 6,
        "question": "What are the symptoms of hives?",
        "relevant_sources": [
            "hives.txt"
        ]
    },

    {
        "id": 7,
        "question": "What is amoxicillin used for?",
        "relevant_sources": [
            "amoxicillin.txt"
        ]
    },

    {
        "id": 8,
        "question": "What is amlodipine used to treat?",
        "relevant_sources": [
            "amlodipine.txt"
        ]
    },

    {
        "id": 9,
        "question": "What is allopurinol used to treat?",
        "relevant_sources": [
            "allopurinol-zyloric_uricto.txt"
        ]
    },

    {
        "id": 10,
        "question": "What is adalimumab used to treat?",
        "relevant_sources": [
            "adalimumab.txt"
        ]
    }

]


# ============================================================
# DISPLAY EVALUATION DATASET
# ============================================================

print("=" * 70)
print("FORMAL RETRIEVAL EVALUATION DATASET")
print("=" * 70)

print(
    f"Number of evaluation questions: "
    f"{len(evaluation_questions)}"
)

print("\nQuestions:")

for item in evaluation_questions:

    print(
        f"{item['id']:02d}. "
        f"{item['question']}"
    )

    print(
        f"    Relevant source: "
        f"{', '.join(item['relevant_sources'])}"
    )

print("=" * 70)

In [ ]:
# ============================================================
# STEP 29: VALIDATE EVALUATION SOURCE FILENAMES
# ============================================================
# This cell checks whether every ground-truth source filename
# used in the formal evaluation actually exists in the NHS
# metadata.
#
# We do NOT calculate evaluation metrics until this check passes.
# ============================================================

# ------------------------------------------------------------
# Collect all source filenames from the loaded metadata
# ------------------------------------------------------------

available_sources = set()

for model_name in MODEL_NAMES:

    for item in faiss_metadata[model_name]:

        available_sources.add(
            item["source_file"]
        )


# ------------------------------------------------------------
# Validate every evaluation question
# ------------------------------------------------------------

validation_passed = True

print("=" * 70)
print("EVALUATION SOURCE VALIDATION")
print("=" * 70)

print(
    f"Unique NHS source files found: "
    f"{len(available_sources)}"
)

print("\n")


for item in evaluation_questions:

    print(
        f"Question {item['id']:02d}: "
        f"{item['question']}"
    )

    for source in item["relevant_sources"]:

        if source in available_sources:

            print(
                f"  {source} : FOUND"
            )

        else:

            print(
                f"  {source} : NOT FOUND"
            )

            validation_passed = False

    print()


# ============================================================
# FINAL RESULT
# ============================================================

print("=" * 70)

if validation_passed:

    print(
        "SOURCE VALIDATION PASSED"
    )

    print(
        "All evaluation source filenames exist in the metadata."
    )

else:

    print(
        "SOURCE VALIDATION FAILED"
    )

    print(
        "Do NOT calculate retrieval metrics yet."
    )

    print(
        "Replace incorrect filenames using the actual metadata."
    )

print("=" * 70)

In [ ]:
# ============================================================
# STEP 30: CORRECTED FORMAL RETRIEVAL EVALUATION
# ============================================================
# IMPORTANT:
# Ground truth is defined at SOURCE-FILE level.
#
# Multiple retrieved chunks from the same source count only
# once as a relevant source.
#
# Therefore:
#   Precision@5 <= 1
#   Recall@5    <= 1
#   MRR         <= 1
#   nDCG@5      <= 1
# ============================================================

import math
import pandas as pd


EVALUATION_TOP_K = 5


# ============================================================
# SOURCE-LEVEL METRICS
# ============================================================

def precision_at_k_source_level(
    retrieved_sources,
    relevant_sources,
    k
):

    retrieved = retrieved_sources[:k]

    # Count each source only once
    unique_retrieved = list(
        dict.fromkeys(retrieved)
    )

    relevant_retrieved = sum(
        1
        for source in unique_retrieved
        if source in relevant_sources
    )

    return relevant_retrieved / k


def recall_at_k_source_level(
    retrieved_sources,
    relevant_sources,
    k
):

    retrieved = retrieved_sources[:k]

    # Remove duplicate source files
    unique_retrieved = set(retrieved)

    relevant_retrieved = len(
        unique_retrieved.intersection(
            relevant_sources
        )
    )

    return (
        relevant_retrieved /
        len(relevant_sources)
    )


def reciprocal_rank_source_level(
    retrieved_sources,
    relevant_sources
):

    # First occurrence of a relevant source
    for rank, source in enumerate(
        retrieved_sources,
        start=1
    ):

        if source in relevant_sources:

            return 1.0 / rank

    return 0.0


def ndcg_at_k_source_level(
    retrieved_sources,
    relevant_sources,
    k
):

    retrieved = retrieved_sources[:k]

    # --------------------------------------------------------
    # Important:
    # Only the FIRST occurrence of a source can receive
    # relevance = 1.
    #
    # Duplicate chunks from the same source are not counted
    # repeatedly.
    # --------------------------------------------------------

    seen_sources = set()

    relevance_scores = []

    for source in retrieved:

        if (
            source in relevant_sources
            and source not in seen_sources
        ):

            relevance_scores.append(1)

            seen_sources.add(source)

        else:

            relevance_scores.append(0)


    # --------------------------------------------------------
    # DCG
    # --------------------------------------------------------

    dcg = 0.0

    for rank, relevance in enumerate(
        relevance_scores,
        start=1
    ):

        dcg += (
            relevance /
            math.log2(rank + 1)
        )


    # --------------------------------------------------------
    # Ideal DCG
    # --------------------------------------------------------

    ideal_relevant = min(
        len(relevant_sources),
        k
    )

    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(
            1,
            ideal_relevant + 1
        )
    )


    if idcg == 0:

        return 0.0

    return dcg / idcg


# ============================================================
# RUN EVALUATION
# ============================================================

evaluation_records = []

print("=" * 70)
print("CORRECTED FORMAL RETRIEVAL EVALUATION")
print("=" * 70)


for model_name in MODEL_NAMES:

    print("\n")
    print("-" * 70)
    print(f"MODEL: {model_name}")
    print("-" * 70)


    model_precisions = []
    model_recalls = []
    model_mrrs = []
    model_ndcgs = []


    for item in evaluation_questions:

        question = item["question"]

        relevant_sources = set(
            item["relevant_sources"]
        )


        # ----------------------------------------------------
        # RETRIEVE TOP-K
        # ----------------------------------------------------

        retrieved = retrieve_documents(

            question=question,

            model_name=model_name,

            top_k=EVALUATION_TOP_K
        )


        retrieved_sources = [

            result["source_file"]

            for result in retrieved

        ]


        # ----------------------------------------------------
        # METRICS
        # ----------------------------------------------------

        precision = precision_at_k_source_level(

            retrieved_sources,

            relevant_sources,

            EVALUATION_TOP_K

        )


        recall = recall_at_k_source_level(

            retrieved_sources,

            relevant_sources,

            EVALUATION_TOP_K

        )


        mrr = reciprocal_rank_source_level(

            retrieved_sources,

            relevant_sources

        )


        ndcg = ndcg_at_k_source_level(

            retrieved_sources,

            relevant_sources,

            EVALUATION_TOP_K

        )


        # ----------------------------------------------------
        # STORE
        # ----------------------------------------------------

        model_precisions.append(
            precision
        )

        model_recalls.append(
            recall
        )

        model_mrrs.append(
            mrr
        )

        model_ndcgs.append(
            ndcg
        )


        evaluation_records.append({

            "model": model_name,

            "question_id": item["id"],

            "question": question,

            "precision_at_5": precision,

            "recall_at_5": recall,

            "mrr": mrr,

            "ndcg_at_5": ndcg

        })


    # --------------------------------------------------------
    # MODEL AVERAGES
    # --------------------------------------------------------

    mean_precision = (
        sum(model_precisions) /
        len(model_precisions)
    )

    mean_recall = (
        sum(model_recalls) /
        len(model_recalls)
    )

    mean_mrr = (
        sum(model_mrrs) /
        len(model_mrrs)
    )

    mean_ndcg = (
        sum(model_ndcgs) /
        len(model_ndcgs)
    )


    print(
        f"Mean Precision@5 : "
        f"{mean_precision:.4f}"
    )

    print(
        f"Mean Recall@5    : "
        f"{mean_recall:.4f}"
    )

    print(
        f"Mean MRR         : "
        f"{mean_mrr:.4f}"
    )

    print(
        f"Mean nDCG@5      : "
        f"{mean_ndcg:.4f}"
    )


# ============================================================
# DATAFRAME
# ============================================================

retrieval_results_df = pd.DataFrame(
    evaluation_records
)


# ============================================================
# FINAL VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("METRIC RANGE VALIDATION")
print("=" * 70)

metric_columns = [

    "precision_at_5",
    "recall_at_5",
    "mrr",
    "ndcg_at_5"

]

for column in metric_columns:

    minimum = retrieval_results_df[
        column
    ].min()

    maximum = retrieval_results_df[
        column
    ].max()

    print(
        f"{column:20} "
        f"min={minimum:.4f} "
        f"max={maximum:.4f}"
    )

    if minimum < 0 or maximum > 1:

        raise ValueError(
            f"{column} is outside [0,1]."
        )


print("\n")
print("=" * 70)
print("CORRECTED RETRIEVAL EVALUATION COMPLETED")
print("=" * 70)

print(
    f"Models evaluated   : {len(MODEL_NAMES)}"
)

print(
    f"Questions evaluated: "
    f"{len(evaluation_questions)}"
)

print(
    f"Total evaluations  : "
    f"{len(retrieval_results_df)}"
)

print("=" * 70)

In [ ]:
# ============================================================
# STEP 31: SAVE CORRECTED RETRIEVAL RESULTS
# ============================================================

import json
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Healthcare_RAG"
)

RESULTS_DIR = PROJECT_DIR / "results"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Save question-level results
# ------------------------------------------------------------

question_results_path = (
    RESULTS_DIR /
    "corrected_retrieval_results.csv"
)

retrieval_results_df.to_csv(
    question_results_path,
    index=False
)


# ------------------------------------------------------------
# Create model-level summary
# ------------------------------------------------------------

retrieval_summary_df = (
    retrieval_results_df
    .groupby("model")
    [
        [
            "precision_at_5",
            "recall_at_5",
            "mrr",
            "ndcg_at_5"
        ]
    ]
    .mean()
    .reset_index()
)


# ------------------------------------------------------------
# Rename columns for report readability
# ------------------------------------------------------------

retrieval_summary_df = retrieval_summary_df.rename(

    columns={

        "model": "Embedding Model",

        "precision_at_5": "Precision@5",

        "recall_at_5": "Recall@5",

        "mrr": "MRR",

        "ndcg_at_5": "nDCG@5"

    }
)


# ------------------------------------------------------------
# Save summary
# ------------------------------------------------------------

summary_path = (
    RESULTS_DIR /
    "retrieval_model_comparison.csv"
)

retrieval_summary_df.to_csv(
    summary_path,
    index=False
)


# ------------------------------------------------------------
# Display final table
# ------------------------------------------------------------

print("=" * 70)
print("CORRECTED RETRIEVAL RESULTS SAVED")
print("=" * 70)

print("\nModel Comparison:")
print(
    retrieval_summary_df.to_string(
        index=False
    )
)

print("\n")
print(f"Question-level results:")
print(question_results_path)

print(f"\nModel-level summary:")
print(summary_path)

print("=" * 70)

In [ ]:
# ============================================================
# STEP 32: RETRIEVAL MODEL COMPARISON
# ============================================================
# Creates:
#   1. Clean comparison table
#   2. Bar chart comparing the four metrics
#
# Uses ONLY the corrected retrieval results.
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# RESULTS DIRECTORY
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Healthcare_RAG"
)

RESULTS_DIR = PROJECT_DIR / "results"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# LOAD CORRECTED RESULTS
# ------------------------------------------------------------

summary_path = (
    RESULTS_DIR /
    "retrieval_model_comparison.csv"
)

comparison_df = pd.read_csv(
    summary_path
)


# ------------------------------------------------------------
# ROUND VALUES FOR REPORT
# ------------------------------------------------------------

report_table = comparison_df.copy()

metric_columns = [
    "Precision@5",
    "Recall@5",
    "MRR",
    "nDCG@5"
]

for column in metric_columns:

    report_table[column] = (
        report_table[column]
        .round(4)
    )


# ------------------------------------------------------------
# DISPLAY TABLE
# ------------------------------------------------------------

print("=" * 70)
print("RETRIEVAL MODEL COMPARISON")
print("=" * 70)

print(
    report_table.to_string(
        index=False
    )
)

print("=" * 70)


# ------------------------------------------------------------
# SAVE REPORT TABLE
# ------------------------------------------------------------

table_path = (
    RESULTS_DIR /
    "retrieval_comparison_report_table.csv"
)

report_table.to_csv(
    table_path,
    index=False
)


# ============================================================
# CREATE BAR CHART
# ============================================================

plot_df = report_table.set_index(
    "Embedding Model"
)

ax = plot_df[
    metric_columns
].plot(
    kind="bar",
    figsize=(10, 6)
)

ax.set_title(
    "Comparison of Embedding Models for Healthcare Retrieval"
)

ax.set_xlabel(
    "Embedding Model"
)

ax.set_ylabel(
    "Score"
)

ax.set_ylim(
    0,
    1.05
)

ax.legend(
    title="Metric"
)

plt.xticks(
    rotation=0
)

plt.tight_layout()


# ------------------------------------------------------------
# SAVE FIGURE
# ------------------------------------------------------------

figure_path = (
    RESULTS_DIR /
    "retrieval_model_comparison.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# CONFIRMATION
# ============================================================

print("\n")
print("=" * 70)
print("REPORT OUTPUTS CREATED")
print("=" * 70)

print(
    f"Table : {table_path}"
)

print(
    f"Graph : {figure_path}"
)

print("=" * 70)

In [ ]:
# ============================================================
# STEP 33: GENERATE FORMAL RAG EVALUATION RESPONSES
# ============================================================
# Best retrieval model from the corrected evaluation:
#     MPNet
#
# Pipeline:
#     Question
#        ↓
#     MPNet
#        ↓
#     FAISS
#        ↓
#     Top-5 NHS chunks
#        ↓
#     Llama 3.1 8B Instruct
#
# These outputs will later be used for RAGAS evaluation.
# ============================================================

import json
from pathlib import Path
import pandas as pd
import time


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

RAGAS_EMBEDDING_MODEL = "MPNet"
RAGAS_TOP_K = 5


# ------------------------------------------------------------
# RESULTS DIRECTORY
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Healthcare_RAG"
)

RESULTS_DIR = PROJECT_DIR / "results"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# GENERATE RESPONSES
# ------------------------------------------------------------

formal_rag_results = []

print("=" * 70)
print("FORMAL RAG RESPONSE GENERATION")
print("=" * 70)

print(
    f"Embedding Model : {RAGAS_EMBEDDING_MODEL}"
)

print(
    f"Top-K            : {RAGAS_TOP_K}"
)

print(
    "Generator        : Llama 3.1 8B Instruct"
)

print(
    f"Questions         : {len(evaluation_questions)}"
)

print("=" * 70)


for number, item in enumerate(
    evaluation_questions,
    start=1
):

    question = item["question"]

    print(
        f"\n[{number}/{len(evaluation_questions)}] "
        f"{question}"
    )

    start_time = time.time()


    # --------------------------------------------------------
    # RUN RAG
    # --------------------------------------------------------

    result = generate_llama_rag_answer(

        question=question,

        embedding_model_name=RAGAS_EMBEDDING_MODEL,

        top_k=RAGAS_TOP_K,

        max_new_tokens=180
    )


    total_time = (
        time.time() - start_time
    )


    # --------------------------------------------------------
    # EXTRACT CONTEXT
    # --------------------------------------------------------

    contexts = [

        document["text"]

        for document
        in result["retrieved_documents"]

    ]


    # --------------------------------------------------------
    # EXTRACT SOURCES
    # --------------------------------------------------------

    sources = [

        document["source_file"]

        for document
        in result["retrieved_documents"]

    ]


    # --------------------------------------------------------
    # SAVE RESULT
    # --------------------------------------------------------

    formal_rag_results.append({

        "question_id":
            item["id"],

        "question":
            question,

        "answer":
            result["answer"],

        "contexts":
            contexts,

        "retrieved_sources":
            sources,

        "relevant_sources":
            item["relevant_sources"],

        "embedding_model":
            RAGAS_EMBEDDING_MODEL,

        "generator_model":
            "meta-llama/Llama-3.1-8B-Instruct",

        "top_k":
            RAGAS_TOP_K,

        "retrieval_time_seconds":
            result["retrieval_time"],

        "generation_time_seconds":
            result["generation_time"],

        "total_time_seconds":
            total_time

    })


    print(
        f"Completed in {total_time:.2f} seconds"
    )


# ============================================================
# SAVE JSON
# ============================================================

formal_rag_path = (
    RESULTS_DIR /
    "formal_rag_evaluation_data.json"
)

with open(
    formal_rag_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        formal_rag_results,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# CREATE SUMMARY TABLE
# ============================================================

formal_rag_summary = pd.DataFrame([

    {
        "question_id":
            result["question_id"],

        "question":
            result["question"],

        "answer":
            result["answer"],

        "retrieval_time_seconds":
            result["retrieval_time_seconds"],

        "generation_time_seconds":
            result["generation_time_seconds"],

        "total_time_seconds":
            result["total_time_seconds"]

    }

    for result in formal_rag_results

])


# ------------------------------------------------------------
# SAVE CSV
# ------------------------------------------------------------

formal_rag_csv = (
    RESULTS_DIR /
    "formal_rag_evaluation_summary.csv"
)

formal_rag_summary.to_csv(
    formal_rag_csv,
    index=False
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n")
print("=" * 70)
print("FORMAL RAG RESPONSE GENERATION COMPLETED")
print("=" * 70)

print(
    f"Responses generated : "
    f"{len(formal_rag_results)}"
)

print(
    f"JSON file           : "
    f"{formal_rag_path}"
)

print(
    f"CSV file            : "
    f"{formal_rag_csv}"
)

print("=" * 70)

In [ ]:
# ============================================================
# STEP 34: PREPARE RAGAS EVALUATION DATASET
# ============================================================

import json
from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Healthcare_RAG"
)

RESULTS_DIR = PROJECT_DIR / "results"

formal_rag_path = (
    RESULTS_DIR /
    "formal_rag_evaluation_data.json"
)


# ------------------------------------------------------------
# LOAD SAVED RESULTS
# ------------------------------------------------------------

if not formal_rag_path.exists():

    raise FileNotFoundError(
        f"Could not find:\n{formal_rag_path}\n\n"
        "Run Cell 33 first."
    )


with open(
    formal_rag_path,
    "r",
    encoding="utf-8"
) as file:

    formal_rag_results = json.load(file)


# ------------------------------------------------------------
# BUILD RAGAS DATASET
# ------------------------------------------------------------

ragas_records = []

for result in formal_rag_results:

    ragas_records.append({

        "user_input":
            result["question"],

        "response":
            result["answer"],

        "retrieved_contexts":
            result["contexts"],

        "reference":
            result["contexts"][0]
            if result["contexts"]
            else ""

    })


ragas_df = pd.DataFrame(
    ragas_records
)


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("=" * 70)
print("RAGAS DATASET PREPARED")
print("=" * 70)

print(
    f"Evaluation samples: "
    f"{len(ragas_df)}"
)

print(
    f"Columns: "
    f"{list(ragas_df.columns)}"
)

print("\nFirst question:")
print(
    ragas_df.iloc[0]["user_input"]
)

print("\nFirst response:")
print(
    ragas_df.iloc[0]["response"]
)

print("\nNumber of retrieved contexts:")
print(
    len(ragas_df.iloc[0]["retrieved_contexts"])
)

print("=" * 70)

In [ ]:
# ============================================================
# STEP 35: CREATE REFERENCE ANSWERS FOR RAGAS
# ============================================================
# IMPORTANT:
# These are reference answers derived from the known relevant
# NHS source documents.
#
# They are NOT generated by Llama.
# They are NOT taken from the retrieved results.
#
# This keeps the RAGAS evaluation separate from retrieval.
# ============================================================

import json
from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Healthcare_RAG"
)

RESULTS_DIR = PROJECT_DIR / "results"


formal_rag_path = (
    RESULTS_DIR /
    "formal_rag_evaluation_data.json"
)


# ------------------------------------------------------------
# LOAD FORMAL RAG RESULTS
# ------------------------------------------------------------

with open(
    formal_rag_path,
    "r",
    encoding="utf-8"
) as file:

    formal_rag_results = json.load(file)


# ============================================================
# REFERENCE ANSWERS
# ============================================================
# These should be based on the actual NHS source content.
#
# We will keep them concise because they are used as reference
# answers for answer-relevance evaluation.
# ============================================================

reference_answers = {

    1: (
        "The main symptoms of asthma include wheezing, "
        "coughing, shortness of breath, and a feeling of "
        "tightness in the chest. Symptoms can be mild or "
        "severe and may be worse at night or early in the morning."
    ),

    2: (
        "Asthma symptoms occur when the airways become swollen "
        "and narrower. Symptoms may be triggered by allergies, "
        "exercise, smoke, cold air, or infections."
    ),

    3: (
        "Asthma is assessed using the symptoms and medical "
        "history, together with tests that check how well "
        "the lungs work."
    ),

    4: (
        "Aciclovir is an antiviral medicine used to treat "
        "viral infections caused by herpes viruses."
    ),

    5: (
        "Food allergy symptoms can include skin reactions, "
        "swelling, breathing difficulties, digestive symptoms, "
        "and other allergic reactions."
    ),

    6: (
        "Hives cause an itchy rash that can appear as raised "
        "and swollen patches or bumps on the skin."
    ),

    7: (
        "Amoxicillin is a penicillin-type antibiotic used "
        "to treat certain bacterial infections."
    ),

    8: (
        "Amlodipine is a medicine used to treat high blood "
        "pressure and some heart-related conditions."
    ),

    9: (
        "Allopurinol is used to reduce uric acid levels and "
        "is commonly used to prevent gout attacks."
    ),

    10: (
        "Adalimumab is a biological medicine used to treat "
        "several inflammatory conditions."
    )

}


# ============================================================
# VALIDATE REFERENCE ANSWERS
# ============================================================

missing_answers = []

for item in formal_rag_results:

    question_id = item["question_id"]

    if question_id not in reference_answers:

        missing_answers.append(
            question_id
        )


if missing_answers:

    raise ValueError(
        "Missing reference answers for question IDs: "
        f"{missing_answers}"
    )


# ============================================================
# BUILD FINAL RAGAS DATA
# ============================================================

ragas_records = []

for item in formal_rag_results:

    question_id = item["question_id"]

    ragas_records.append({

        "user_input":
            item["question"],

        "response":
            item["answer"],

        "retrieved_contexts":
            item["contexts"],

        "reference":
            reference_answers[question_id]

    })


ragas_df = pd.DataFrame(
    ragas_records
)


# ============================================================
# SAVE
# ============================================================

ragas_path = (
    RESULTS_DIR /
    "ragas_evaluation_dataset.json"
)

with open(
    ragas_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        ragas_records,
        file,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# DISPLAY
# ============================================================

print("=" * 70)
print("RAGAS DATASET CREATED")
print("=" * 70)

print(
    f"Evaluation questions : "
    f"{len(ragas_records)}"
)

print(
    f"Reference answers    : "
    f"{len(reference_answers)}"
)

print(
    f"Saved file           : "
    f"{ragas_path}"
)

print("\nSample:")
print(
    ragas_df[
        [
            "user_input",
            "reference"
        ]
    ].head(3).to_string(
        index=False
    )
)

print("=" * 70)

In [ ]:
# ============================================================
# STEP 36: VERIFY RAGAS REFERENCE ANSWERS
# ============================================================
# This cell compares each reference answer with the actual
# relevant NHS source text stored in the metadata.
#
# DO NOT calculate RAGAS yet.
#
# We want to inspect the source evidence first.
# ============================================================

import json
from pathlib import Path


# ------------------------------------------------------------
# PROJECT PATH
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Healthcare_RAG"
)

FAISS_DIR = (
    PROJECT_DIR /
    "data" /
    "faiss"
)

RESULTS_DIR = (
    PROJECT_DIR /
    "results"
)


# ------------------------------------------------------------
# LOAD BGE METADATA
# ------------------------------------------------------------
# BGE is sufficient here because all models use the same
# underlying NHS chunk collection.
# ------------------------------------------------------------

metadata_path = (
    FAISS_DIR /
    "BGE" /
    "metadata.json"
)

with open(
    metadata_path,
    "r",
    encoding="utf-8"
) as file:

    metadata = json.load(file)


# ------------------------------------------------------------
# BUILD SOURCE -> CHUNKS MAPPING
# ------------------------------------------------------------

source_chunks = {}

for item in metadata:

    source = item["source_file"]

    if source not in source_chunks:

        source_chunks[source] = []

    source_chunks[source].append(
        item
    )


# ============================================================
# DISPLAY REFERENCE ANSWER + SOURCE EVIDENCE
# ============================================================

print("=" * 70)
print("RAGAS REFERENCE ANSWER VERIFICATION")
print("=" * 70)


for item in evaluation_questions:

    question_id = item["id"]

    question = item["question"]

    relevant_sources = item[
        "relevant_sources"
    ]


    print("\n")
    print("=" * 70)

    print(
        f"QUESTION {question_id}: "
        f"{question}"
    )

    print("=" * 70)


    # --------------------------------------------------------
    # REFERENCE ANSWER
    # --------------------------------------------------------

    print("\nREFERENCE ANSWER:")
    print(
        reference_answers[question_id]
    )


    # --------------------------------------------------------
    # SOURCE EVIDENCE
    # --------------------------------------------------------

    print("\nRELEVANT NHS SOURCE TEXT:")

    for source in relevant_sources:

        print("\n")
        print(
            f"SOURCE: {source}"
        )

        if source not in source_chunks:

            print(
                "WARNING: Source not found in metadata."
            )

            continue


        # Display the first few chunks belonging to the
        # relevant source.
        #
        # This gives us the actual NHS text to inspect.
        # ----------------------------------------------------

        chunks = source_chunks[source]

        for chunk in chunks[:3]:

            print(
                f"\nChunk ID: {chunk['chunk_id']}"
            )

            print(
                chunk["text"][:2000]
            )

            print(
                "\n" + "-" * 50
            )


print("\n")
print("=" * 70)
print("REFERENCE VERIFICATION DISPLAY COMPLETED")
print("=" * 70)

print(
    "Review the displayed NHS evidence before running RAGAS."
)

print("=" * 70)

In [ ]:
# ============================================================
# STEP 37: INSTALL AND CONFIGURE RAGAS
# ============================================================

!pip install -q -U ragas datasets

print("=" * 70)
print("RAGAS INSTALLATION COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================
# STEP 38A: FIX RAGAS DEPENDENCIES
# ============================================================

!pip install -q -U \
    "langchain-community" \
    "langchain-core" \
    "instructor"

print("=" * 70)
print("RAGAS DEPENDENCIES INSTALLED")
print("=" * 70)

print("IMPORTANT: Restart the Colab runtime after this cell.")
print("=" * 70)

In [ ]:
# ============================================================
# STEP 38B: SAVE RESPONSE-QUALITY EVALUATION DATA
# ============================================================

from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Healthcare_RAG"
)

RESULTS_DIR = PROJECT_DIR / "results"

formal_rag_path = (
    RESULTS_DIR /
    "formal_rag_evaluation_data.json"
)

if not formal_rag_path.exists():

    raise FileNotFoundError(
        "formal_rag_evaluation_data.json was not found."
    )


with open(
    formal_rag_path,
    "r",
    encoding="utf-8"
) as file:

    results = json.load(file)


# ------------------------------------------------------------
# CREATE RESPONSE QUALITY TABLE
# ------------------------------------------------------------

response_quality_df = pd.DataFrame([

    {
        "Question ID":
            item["question_id"],

        "Question":
            item["question"],

        "Generated Answer":
            item["answer"],

        "Retrieved Sources":
            ", ".join(
                item["retrieved_sources"]
            ),

        "Retrieval Time (s)":
            round(
                item["retrieval_time_seconds"],
                4
            ),

        "Generation Time (s)":
            round(
                item["generation_time_seconds"],
                4
            ),

        "Total Time (s)":
            round(
                item["total_time_seconds"],
                4
            )

    }

    for item in results

])


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

output_path = (
    RESULTS_DIR /
    "response_quality_evaluation.csv"
)

response_quality_df.to_csv(
    output_path,
    index=False
)


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("=" * 70)
print("RESPONSE QUALITY DATASET SAVED")
print("=" * 70)

print(
    f"Evaluation cases : {len(response_quality_df)}"
)

print(
    f"Generator        : Llama 3.1 8B Instruct"
)

print(
    f"Embedding        : MPNet"
)

print(
    f"Output           : {output_path}"
)

print("=" * 70)

In [ ]:
# ============================================================
# STEP 39: RESPONSE-TIME STATISTICS
# ============================================================

import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# LOAD RESULTS
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Healthcare_RAG"
)

RESULTS_DIR = PROJECT_DIR / "results"

input_path = (
    RESULTS_DIR /
    "response_quality_evaluation.csv"
)

df = pd.read_csv(
    input_path
)


# ============================================================
# CALCULATE STATISTICS
# ============================================================

retrieval_mean = (
    df["Retrieval Time (s)"].mean()
)

retrieval_std = (
    df["Retrieval Time (s)"].std()
)

generation_mean = (
    df["Generation Time (s)"].mean()
)

generation_std = (
    df["Generation Time (s)"].std()
)

total_mean = (
    df["Total Time (s)"].mean()
)

total_std = (
    df["Total Time (s)"].std()
)

total_min = (
    df["Total Time (s)"].min()
)

total_max = (
    df["Total Time (s)"].max()
)


# ============================================================
# CREATE SUMMARY
# ============================================================

performance_summary = pd.DataFrame({

    "Metric": [

        "Retrieval Time",

        "Generation Time",

        "Total Response Time"

    ],

    "Mean (seconds)": [

        retrieval_mean,

        generation_mean,

        total_mean

    ],

    "Std Dev (seconds)": [

        retrieval_std,

        generation_std,

        total_std

    ],

    "Minimum (seconds)": [

        df["Retrieval Time (s)"].min(),

        df["Generation Time (s)"].min(),

        total_min

    ],

    "Maximum (seconds)": [

        df["Retrieval Time (s)"].max(),

        df["Generation Time (s)"].max(),

        total_max

    ]

})


# ============================================================
# DISPLAY
# ============================================================

print("=" * 70)
print("RAG RESPONSE-TIME PERFORMANCE")
print("=" * 70)

print(
    performance_summary.to_string(
        index=False
    )
)

print("=" * 70)


# ============================================================
# SAVE
# ============================================================

performance_path = (
    RESULTS_DIR /
    "rag_performance_summary.csv"
)

performance_summary.to_csv(
    performance_path,
    index=False
)


print(
    f"\nSaved to:\n{performance_path}"
)

In [ ]:
# ============================================================
# STEP 40: CREATE FINAL EXPERIMENTAL RESULTS
# ============================================================

from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Healthcare_RAG"
)

RESULTS_DIR = PROJECT_DIR / "results"


# ============================================================
# LOAD RETRIEVAL RESULTS
# ============================================================

retrieval_path = (
    RESULTS_DIR /
    "retrieval_model_comparison.csv"
)

retrieval_df = pd.read_csv(
    retrieval_path
)


print("=" * 70)
print("LOADED RETRIEVAL RESULTS")
print("=" * 70)

print(
    retrieval_df.to_string(
        index=False
    )
)


# ============================================================
# VALIDATE REQUIRED COLUMNS
# ============================================================

required_columns = [

    "Embedding Model",
    "Precision@5",
    "Recall@5",
    "MRR",
    "nDCG@5"

]

missing_columns = [

    column

    for column in required_columns

    if column not in retrieval_df.columns

]


if missing_columns:

    raise ValueError(
        "Missing columns: "
        f"{missing_columns}"
    )


# ============================================================
# LOAD RAG PERFORMANCE
# ============================================================

performance_path = (
    RESULTS_DIR /
    "rag_performance_summary.csv"
)

performance_df = pd.read_csv(
    performance_path
)


# ============================================================
# FIND BEST RETRIEVAL MODEL
# ============================================================
# Primary criterion:
#     Highest MRR
#
# Secondary criterion:
#     Highest nDCG@5
# ============================================================

best_model = (

    retrieval_df

    .sort_values(
        by=[
            "MRR",
            "nDCG@5"
        ],

        ascending=False
    )

    .iloc[0]["Embedding Model"]

)


# ============================================================
# EXTRACT PERFORMANCE VALUES
# ============================================================

mean_retrieval_time = (

    performance_df.loc[

        performance_df["Metric"]
        == "Retrieval Time",

        "Mean (seconds)"

    ].iloc[0]

)


mean_generation_time = (

    performance_df.loc[

        performance_df["Metric"]
        == "Generation Time",

        "Mean (seconds)"

    ].iloc[0]

)


mean_total_time = (

    performance_df.loc[

        performance_df["Metric"]
        == "Total Response Time",

        "Mean (seconds)"

    ].iloc[0]

)


# ============================================================
# CREATE FINAL RETRIEVAL TABLE
# ============================================================

final_results = retrieval_df.copy()

final_results["Generator"] = (
    "Llama 3.1 8B Instruct"
)

final_results["Evaluation Top-K"] = 5


# ============================================================
# DISPLAY RETRIEVAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RETRIEVAL RESULTS")
print("=" * 70)

print(
    final_results.to_string(
        index=False
    )
)


# ============================================================
# DISPLAY BEST MODEL
# ============================================================

print("\n")
print("=" * 70)
print("BEST RETRIEVAL MODEL")
print("=" * 70)

print(
    f"Best model: {best_model}"
)

print(
    "Primary criterion : MRR"
)

print(
    "Secondary criterion: nDCG@5"
)


# ============================================================
# DISPLAY RAG PERFORMANCE
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RAG PERFORMANCE")
print("=" * 70)

print(
    "Generator           : "
    "Llama 3.1 8B Instruct"
)

print(
    f"Embedding Model     : "
    f"{best_model}"
)

print(
    f"Mean Retrieval Time : "
    f"{mean_retrieval_time:.4f} seconds"
)

print(
    f"Mean Generation Time: "
    f"{mean_generation_time:.4f} seconds"
)

print(
    f"Mean Total Response : "
    f"{mean_total_time:.4f} seconds"
)


# ============================================================
# SAVE FINAL RETRIEVAL RESULTS
# ============================================================

final_results_path = (
    RESULTS_DIR /
    "final_retrieval_results.csv"
)

final_results.to_csv(
    final_results_path,
    index=False
)


# ============================================================
# CREATE EXPERIMENT SUMMARY
# ============================================================

experiment_summary = pd.DataFrame({

    "Item": [

        "Best Retrieval Model",

        "Generator",

        "Top-K",

        "Mean Retrieval Time (s)",

        "Mean Generation Time (s)",

        "Mean Total Response Time (s)"

    ],

    "Value": [

        best_model,

        "Llama 3.1 8B Instruct",

        5,

        round(
            mean_retrieval_time,
            4
        ),

        round(
            mean_generation_time,
            4
        ),

        round(
            mean_total_time,
            4
        )

    ]

})


# ============================================================
# SAVE EXPERIMENT SUMMARY
# ============================================================

summary_path = (
    RESULTS_DIR /
    "final_experiment_summary.csv"
)

experiment_summary.to_csv(
    summary_path,
    index=False
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL EXPERIMENTAL RESULTS SAVED")
print("=" * 70)

print(
    f"Retrieval results : "
    f"{final_results_path}"
)

print(
    f"Experiment summary: "
    f"{summary_path}"
)

print("=" * 70)

In [ ]:
# ============================================================
# STEP 41: FINAL CHATBOT CONFIGURATION
# ============================================================
# Final selected system:
#
# Retrieval  : MPNet + FAISS
# Generation : Llama 3.1 8B Instruct
# Knowledge  : NHS
# Top-K      : 5
# ============================================================

FINAL_EMBEDDING_MODEL = "MPNet"
FINAL_TOP_K = 5

print("=" * 70)
print("FINAL CHATBOT CONFIGURATION")
print("=" * 70)

print(
    f"Embedding Model : {FINAL_EMBEDDING_MODEL}"
)

print(
    f"Generator        : Llama 3.1 8B Instruct"
)

print(
    f"Retriever         : FAISS"
)

print(
    f"Knowledge Base    : NHS"
)

print(
    f"Top-K             : {FINAL_TOP_K}"
)

print("=" * 70)

In [32]:
# ============================================================
# STEP 42: FINAL MPNet + LLAMA CHATBOT
# ============================================================

import gradio as gr
import time


# ============================================================
# FINAL CHAT FUNCTION
# ============================================================

def final_chatbot(question):

    # --------------------------------------------------------
    # Validate question
    # --------------------------------------------------------

    if not question or not question.strip():

        return (
            "Please enter a healthcare question.",
            "",
            ""
        )


    # --------------------------------------------------------
    # Start timing
    # --------------------------------------------------------

    start_time = time.time()


    # ========================================================
    # RUN FINAL RAG PIPELINE
    # ========================================================

    result = generate_llama_rag_answer(

        question=question.strip(),

        embedding_model_name=FINAL_EMBEDDING_MODEL,

        top_k=FINAL_TOP_K,

        max_new_tokens=180

    )


    # --------------------------------------------------------
    # Total response time
    # --------------------------------------------------------

    total_time = (
        time.time() - start_time
    )


    # ========================================================
    # FORMAT RETRIEVED SOURCES
    # ========================================================

    source_lines = []


    for document in result["retrieved_documents"]:

        source_lines.append(

            f"Rank {document['rank']} | "
            f"Source: {document['source_file']} | "
            f"Chunk: {document['chunk_id']} | "
            f"Similarity: "
            f"{document['score']:.4f}"

        )


    sources_text = "\n".join(
        source_lines
    )


    # ========================================================
    # FORMAT PERFORMANCE
    # ========================================================

    performance_text = (

        "Embedding Model: "
        f"{FINAL_EMBEDDING_MODEL}\n"

        "Generator: "
        "Llama 3.1 8B Instruct\n"

        "Retriever: FAISS\n"

        f"Top-K: {FINAL_TOP_K}\n"

        f"Retrieval Time: "
        f"{result['retrieval_time']:.4f} seconds\n"

        f"Generation Time: "
        f"{result['generation_time']:.4f} seconds\n"

        f"Total Response Time: "
        f"{total_time:.4f} seconds"

    )


    # ========================================================
    # RETURN RESULTS
    # ========================================================

    return (

        result["answer"],

        sources_text,

        performance_text

    )


# ============================================================
# CLEAR FUNCTION
# ============================================================

def clear_final_chat():

    return (
        "",      # Question
        "",      # Answer
        "",      # Sources
        ""       # Performance
    )


# ============================================================
# BUILD GRADIO APPLICATION
# ============================================================

with gr.Blocks(
    title="NHS Healthcare RAG Assistant"
) as final_demo:


    # ========================================================
    # TITLE / DESCRIPTION
    # ========================================================

    gr.Markdown(
        """
        # NHS Healthcare RAG Assistant

        A Retrieval-Augmented Generation system using
        NHS healthcare information, FAISS retrieval,
        MPNet embeddings and Llama 3.1 8B Instruct.
        """
    )


    # ========================================================
    # QUESTION
    # ========================================================

    question_box = gr.Textbox(

        label="Healthcare Question",

        placeholder=(
            "Example: What are the symptoms of asthma?"
        ),

        lines=4

    )


    # ========================================================
    # BUTTONS
    # ========================================================

    with gr.Row():

        ask_button = gr.Button(
            "Ask"
        )

        clear_button = gr.Button(
            "Clear"
        )


    # ========================================================
    # ANSWER
    # ========================================================

    answer_box = gr.Textbox(

        label="Generated Answer",

        lines=10

    )


    # ========================================================
    # SOURCES
    # ========================================================

    sources_box = gr.Textbox(

        label="Retrieved NHS Sources",

        lines=8

    )


    # ========================================================
    # PERFORMANCE
    # ========================================================

    performance_box = gr.Textbox(

        label="System Performance",

        lines=7

    )


    # ========================================================
    # ASK BUTTON
    # ========================================================

    ask_button.click(

        fn=final_chatbot,

        inputs=[
            question_box
        ],

        outputs=[
            answer_box,
            sources_box,
            performance_box
        ]

    )


    # ========================================================
    # CLEAR BUTTON
    # ========================================================

    clear_button.click(

        fn=clear_final_chat,

        inputs=[],

        outputs=[
            question_box,
            answer_box,
            sources_box,
            performance_box
        ]

    )


# ============================================================
# LAUNCH
# ============================================================

final_demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8bc10fed6e18cd15e7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [30]:
# ============================================================
# STEP 43: FINAL CHATBOT VALIDATION
# ============================================================
# Final system:
#
#   MPNet
#      ↓
#   FAISS
#      ↓
#   NHS Context
#      ↓
#   Llama 3.1 8B Instruct
#
# This cell performs final functional tests.
# ============================================================

final_test_questions = [

    "What are the symptoms of asthma?",

    "What are the symptoms of a food allergy?",

    "What is amoxicillin used for?",

    "What is aciclovir used to treat?"

]


final_validation_results = []


print("=" * 70)
print("FINAL CHATBOT VALIDATION")
print("=" * 70)

print(
    f"Embedding Model : {FINAL_EMBEDDING_MODEL}"
)

print(
    "Generator        : Llama 3.1 8B Instruct"
)

print(
    f"Top-K            : {FINAL_TOP_K}"
)

print("=" * 70)


for number, question in enumerate(
    final_test_questions,
    start=1
):

    print("\n")
    print("-" * 70)

    print(
        f"TEST {number}/{len(final_test_questions)}"
    )

    print("-" * 70)

    print(
        f"Question: {question}"
    )

    start_time = time.time()


    # --------------------------------------------------------
    # RUN FINAL SYSTEM
    # --------------------------------------------------------

    result = generate_llama_rag_answer(

        question=question,

        embedding_model_name=FINAL_EMBEDDING_MODEL,

        top_k=FINAL_TOP_K,

        max_new_tokens=180

    )


    elapsed = (
        time.time() - start_time
    )


    # --------------------------------------------------------
    # STORE
    # --------------------------------------------------------

    final_validation_results.append({

        "question":
            question,

        "answer":
            result["answer"],

        "top_source":
            (
                result[
                    "retrieved_documents"
                ][0]["source_file"]
                if result[
                    "retrieved_documents"
                ]
                else "None"
            ),

        "retrieval_time":
            result["retrieval_time"],

        "generation_time":
            result["generation_time"],

        "total_time":
            elapsed

    })


    # --------------------------------------------------------
    # DISPLAY
    # --------------------------------------------------------

    print("\nAnswer:")
    print(
        result["answer"]
    )

    print("\nTop Retrieved Source:")

    if result["retrieved_documents"]:

        top_document = (
            result[
                "retrieved_documents"
            ][0]
        )

        print(
            f"{top_document['source_file']} "
            f"| Similarity: "
            f"{top_document['score']:.4f}"
        )

    else:

        print("No source retrieved.")


    print(
        f"\nTotal Time: "
        f"{elapsed:.4f} seconds"
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL CHATBOT VALIDATION COMPLETED")
print("=" * 70)

print(
    f"Questions tested: "
    f"{len(final_validation_results)}"
)

print(
    "System: MPNet + FAISS + NHS + "
    "Llama 3.1 8B Instruct"
)

print("=" * 70)

FINAL CHATBOT VALIDATION


NameError: name 'FINAL_EMBEDDING_MODEL' is not defined

In [ ]:
# ============================================================
# STEP 44: SAVE FINAL CHATBOT VALIDATION RESULTS
# ============================================================

import json
from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# RESULTS DIRECTORY
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Healthcare_RAG"
)

RESULTS_DIR = PROJECT_DIR / "results"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# SAVE JSON
# ------------------------------------------------------------

json_path = (
    RESULTS_DIR /
    "final_chatbot_validation.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_validation_results,
        file,
        ensure_ascii=False,
        indent=4
    )


# ------------------------------------------------------------
# CREATE DATAFRAME
# ------------------------------------------------------------

validation_df = pd.DataFrame(
    final_validation_results
)


# ------------------------------------------------------------
# SAVE CSV
# ------------------------------------------------------------

csv_path = (
    RESULTS_DIR /
    "final_chatbot_validation.csv"
)

validation_df.to_csv(
    csv_path,
    index=False
)


# ============================================================
# DISPLAY
# ============================================================

print("=" * 70)
print("FINAL CHATBOT VALIDATION RESULTS SAVED")
print("=" * 70)

print(
    f"Questions tested : "
    f"{len(validation_df)}"
)

print(
    "Embedding Model  : MPNet"
)

print(
    "Generator        : Llama 3.1 8B Instruct"
)

print(
    "Retriever        : FAISS"
)

print(
    f"\nJSON : {json_path}"
)

print(
    f"CSV  : {csv_path}"
)

print("=" * 70)

In [ ]:
# ============================================================
# STEP 45: FINAL RESULTS VISUALISATIONS
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Healthcare_RAG"
)

RESULTS_DIR = PROJECT_DIR / "results"


# ------------------------------------------------------------
# LOAD FINAL RESULTS
# ------------------------------------------------------------

results_path = (
    RESULTS_DIR /
    "retrieval_model_comparison.csv"
)

results_df = pd.read_csv(
    results_path
)


# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

metrics = [
    "Precision@5",
    "Recall@5",
    "MRR",
    "nDCG@5"
]


# ============================================================
# CREATE ONE FIGURE PER METRIC
# ============================================================

for metric in metrics:

    plt.figure(
        figsize=(8, 5)
    )

    plt.bar(
        results_df["Embedding Model"],
        results_df[metric]
    )

    plt.title(
        f"{metric} Comparison Across Embedding Models"
    )

    plt.xlabel(
        "Embedding Model"
    )

    plt.ylabel(
        "Score"
    )

    plt.ylim(
        0,
        1.05
    )

    plt.xticks(
        rotation=0
    )

    plt.tight_layout()


    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------

    filename = (
        metric
        .lower()
        .replace("@", "_at_")
        .replace(" ", "_")
        + "_comparison.png"
    )

    output_path = (
        RESULTS_DIR /
        filename
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(
        f"Saved: {output_path}"
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RESULTS VISUALISATIONS CREATED")
print("=" * 70)

print(
    "Four separate 300-DPI figures have been saved."
)

print("=" * 70)